# Notebook 2 — World Bank KPI Selection & Data Retrieval

## Project: Market Expansion Strategy — Travel Company Analysis

### Business Context
A global online travel company needs to identify which 5 countries deserve the most marketing investment. To build a **Market Attractiveness Framework**, we need to measure each country across multiple business dimensions using World Bank data.

### Purpose of This Notebook
This notebook has two connected goals:
1. **KPI Selection** — Search the World Bank indicator catalog to identify and justify the best indicators for each business dimension
2. **Data Retrieval** — Pull actual country-level data for the selected KPIs and prepare clean DataFrames for analysis

In [1]:
# Importing the libraries

import numpy as np
import pandas as pd

In [2]:
import requests
import time

In [3]:
# Load the World Bank indicator catalog created in the API connection notebook.
# This catalog contains available indicators, indicator IDs, source information, and topic classifications.

indicator_catalog_df = pd.read_csv("world_bank_indicator_catalog.csv")

indicator_catalog_df.head()

,id,name,unit,source,sourceNote,sourceOrganization,topics
0,1.0.HCount.1.90usd,Poverty Headcount ($1.90 a day),NaN,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"
1,1.0.HCount.2.5usd,Poverty Headcount ($2.50 a day),NaN,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"
2,1.0.HCount.Mid10to50,Middle Class ($10-50 a day) Headcount,NaN,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"
3,1.0.HCount.Ofcl,Official Moderate Poverty Rate-National,NaN,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of data from Nation...,"[{'id': '11', 'value': 'Poverty '}]"
4,1.0.HCount.Poor4uds,Poverty Headcount ($4 a day),NaN,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"


In [4]:
# Before searching for specific KPIs, we explore the catalog structure to understand its size, columns, and data quality. This helps us plan how to search reliably.
# Check the size of the indicator catalog.
# This confirms how many indicators and metadata fields are available before KPI selection.

indicator_catalog_df.shape

(29501, 7)

In [5]:
# The indicator catalog contains 29,501 indicators and 7 columns.
# This confirms that the World Bank indicator catalog was successfully retrieved and can be used for KPI discovery.

In [6]:
# Review data types and missing values before selecting KPIs.
# This helps identify which metadata fields are reliable for filtering and interpretation.

indicator_catalog_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29501 entries, 0 to 29500
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id                  29501 non-null  object 
 1   name                29501 non-null  object 
 2   unit                0 non-null      float64
 3   source              29501 non-null  object 
 4   sourceNote          14667 non-null  object 
 5   sourceOrganization  14841 non-null  object 
 6   topics              29501 non-null  object 
dtypes: float64(1), object(6)
memory usage: 1.6+ MB


In [7]:
# Checking missing values by column.

indicator_catalog_df.isna().sum()

id                        0
name                      0
unit                  29501
source                    0
sourceNote            14834
sourceOrganization    14660
topics                    0
dtype: int64

In [8]:
# The indicator code, name, source, and topic fields contain no missing values. They are reliable for the KPI selection process.
# The unit field is completely empty; we can drop in the cleaning process.
# The sourceNote and sourceOrganization fields contain missing values, but they are still useful when available because they provide additional context on indicator definitions and data sources.

In [9]:
# Review a random sample of indicator names.
# This helps understand how indicators are named in the World Bank catalog.

indicator_catalog_df["name"].sample(50)

24034    Learning Deprivation Gap;TIMSS 2019 for grade ...
10973    Received government payments: through a mobile...
21121    Wittgenstein Projection: Percentage of the pop...
12880    Labor regulations (% of firms identifying this...
29084    Out-of-school rate for youth of upper secondar...
6432     Gross Ext. Debt Pos., ST Rem., Deposit-Taking ...
20313      Coverage in 3rd quintile (%) - Social Pensions 
6602     Ext. Debt Service Pmt, Deposit-Taking Corp., e...
27890    Percentage of students in lower secondary educ...
2966     Emission Totals - Indirect emissions (N2O) - S...
19302    Benefits incidence in 2nd quintile (%) - Subsi...
7928     CO2 emissions from liquid fuel consumption (% ...
10850    Money put into or taken out of account, poores...
4220     Net bilateral aid flows from DAC donors, Korea...
24082    Learning Deprivation Severity;TIMSS 2007 for g...
13277                    Poverty HCR Estimates (%) - Total
18206    Beneficiary incidence in 1st quintile (poorest.

In [10]:
# Review a random sample of topic classifications.
# Topics can support structured indicator searches later in the notebook.

indicator_catalog_df["topics"].sample(50)

12196                                                   []
25242                                                   []
5839                                                    []
2663                                                    []
15107                                                   []
8197                                                    []
25596                                                   []
16137                                                   []
19741                                                   []
14239                                                   []
6928                                                    []
9402                                                    []
28583                                                   []
8320                                                    []
25586                                                   []
18598    [{'id': '10', 'value': 'Social Protection & La...
2557              [{'id': '20', 'value': 'External Debt'

In [11]:
# The topics field contains predefined World Bank categories.
# These categories can support structured KPI discovery, but the main selection process will rely primarily on indicator names, indicator IDs, and business relevance.

In [12]:
# Check how many unique sources are represented in the catalog.

indicator_catalog_df["source"].nunique()

62

In [13]:
# The World Bank indicator catalog contains data from 62 unique sources.
# This confirms that the catalog aggregates indicators from multiple databases rather than a single centralized dataset.

In [14]:
# Review the largest sources in the catalog.

indicator_catalog_df["source"].value_counts().head(20)

source
{'id': '12', 'value': 'Education Statistics'}                                                   8308
{'id': '29', 'value': 'The Atlas of Social Protection: Indicators of Resilience and Equity'}    3561
{'id': '28', 'value': 'Global Findex database'}                                                 2947
{'id': '22', 'value': 'Quarterly External Debt Statistics SDDS'}                                1800
{'id': '92', 'value': 'Disability Data Hub (DDH)'}                                              1577
{'id': '2', 'value': 'World Development Indicators'}                                            1486
{'id': '57', 'value': 'WDI Database Archives'}                                                  1016
{'id': '14', 'value': 'Gender Statistics'}                                                       928
{'id': '11', 'value': 'Africa Development Indicators'}                                           837
{'id': '86', 'value': 'Global Jobs Indicators Database (JOIN)'}                     

In [15]:
# The indicator catalog is concentrated in several large data sources.
# This confirms that not all indicators are equally relevant for this project.
# KPI selection should therefore be guided by the business case rather than by indicator availability alone.

In [16]:
# Review examples of source notes.

indicator_catalog_df["sourceNote"].sample(10)

4653     The source of non-seasonally adjusted Gross Do...
14468    Percentage of students who were unable to read...
18297    Total transfer amount received by all benefici...
10221                                                  NaN
16271    Gross national income is the total income earn...
27349    The number of persons in the relevant age grou...
8692                                                   NaN
2048     Average years of tertiary schooling, 20-24, to...
1155                                                   NaN
6451                                                   NaN
Name: sourceNote, dtype: object

In [17]:
# The sourceNote field provides detailed indicator definitions when available.
# This field is useful for validating KPI selection, especially when multiple indicators have similar names.

## KPI Selection: Tourism Indicators

### Business Question
Tourism is one of the core dimensions of our Market Attractiveness Framework. A travel company needs to understand both **how many visitors a country attracts** and **how much economic value those visitors generate**.

These two perspectives are captured by two separate indicators:
- **Tourism Arrivals** — measures destination popularity
- **Tourism Receipts** — measures tourism economic value

In [18]:
# Tourism is one of the core dimensions of the business case.
# A travel company is interested not only in the number of visitors a country attracts, but also in the economic value generated by those visitors.
# Therefore, the tourism analysis focuses on two separate questions:

# 1. Which countries attract the highest number of visitors?
# 2. Which countries generate the highest tourism spending?

# These two perspectives will be represented by Tourism Arrivals and Tourism Receipts; per research.

In [19]:
# Search the indicator catalog for tourism-related metrics.
# This provides an initial set of candidate indicators for evaluation.

tourism_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains(
        "tourism",
        case=False,
        na=False
    )
]

tourism_indicators_df[["id", "name"]].head(50)

,id,name
7186,DT.ODA.DACD.PROD.TRSM.CD,"Gross ODA aid disbursement for tourism sector,..."
8524,FC.XPD.TOUR.CR,Tourism and culture function expenditure (in IDR)
27035,ST.INT.ARVL,"International tourism, number of arrivals"
27036,ST.INT.DPRT,"International tourism, number of departures"
27037,ST.INT.RCPT.CD,"International tourism, receipts (current US$)"
27038,ST.INT.RCPT.XP.ZS,"International tourism, receipts (% of total ex..."
27039,ST.INT.TRNR.CD,"International tourism, receipts for passenger ..."
27040,ST.INT.TRNX.CD,"International tourism, expenditures for passen..."
27041,ST.INT.TVLR.CD,"International tourism, receipts for travel ite..."
27042,ST.INT.TVLX.CD,"International tourism, expenditures for travel..."


In [20]:
# The tourism search returns multiple indicators covering arrivals, receipts, expenditures, employment, and tourism-related economic activity.
# For this project, the objective is to evaluate country attractiveness from the perspective of a travel platform.
# Therefore, the focus will be placed on indicators that directly measure tourism demand and tourism value.

In [21]:
# Narrow the tourism indicators to those related to visitor arrivals.

tourism_arrival_candidates = tourism_indicators_df[
    tourism_indicators_df["name"].str.contains(
        "arrivals",
        case=False,
        na=False
    )
]

tourism_arrival_candidates[["id", "name"]]

,id,name
27035,ST.INT.ARVL,"International tourism, number of arrivals"


In [22]:
# Tourism arrivals measure the volume of international visitors entering a country.
# This metric serves as a proxy for destination popularity and overall tourism activity.
# Countries with higher arrival volumes are likely to represent larger tourism markets.

### Tourism KPI — Number of Arrivals
**Selected:** `ST.INT.ARVL`

ST.INT.ARVL (International Tourism Arrivals) was selected as the primary measure of tourism activity. This indicator directly measures inbound visitor volume and aligns closely with the project's objective of identifying attractive travel markets.

In [23]:
selected_tourism_arrivals_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "ST.INT.ARVL"
]

selected_tourism_arrivals_indicator[
    ["id", "name", "sourceNote"]
]

,id,name,sourceNote
27035,ST.INT.ARVL,"International tourism, number of arrivals",International inbound tourists (overnight visi...


### Tourism KPI — Receipts (Current US$)
**Selected:** `ST.INT.RCPT.CD`

ST.INT.RCPT.CD (International Tourism Receipts, Current US$) is selected as the primary measure of tourism value. Together, Tourism Arrivals and Tourism Receipts provide a balanced view of tourism market size and economic impact.

In [24]:
# Narrow the tourism indicators to those related to tourism receipts.

tourism_receipt_candidates = tourism_indicators_df[
    tourism_indicators_df["name"].str.contains(
        "receipts",
        case=False,
        na=False
    )
]

tourism_receipt_candidates[["id", "name"]]

,id,name
27037,ST.INT.RCPT.CD,"International tourism, receipts (current US$)"
27038,ST.INT.RCPT.XP.ZS,"International tourism, receipts (% of total ex..."
27039,ST.INT.TRNR.CD,"International tourism, receipts for passenger ..."
27041,ST.INT.TVLR.CD,"International tourism, receipts for travel ite..."


In [25]:
# Tourism receipts measure the economic value generated by international visitors.
# Unlike arrivals, which measure visitor volume, receipts capture spending and revenue generated through tourism activity.
# This distinction is important because countries with fewer visitors may still generate significant tourism value.

In [26]:
selected_tourism_receipts_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "ST.INT.RCPT.CD"
]

selected_tourism_receipts_indicator[
    ["id", "name", "sourceNote"]
]

,id,name,sourceNote
27037,ST.INT.RCPT.CD,"International tourism, receipts (current US$)",International tourism receipts are expenditure...


In [27]:
# Tourism KPI Summary:

# Two tourism indicators were selected:

#            Business Question                           | Selected Indicator
# Which countries attract the most visitors?             | ST.INT.ARVL
# Which countries generate the highest tourism spending? | ST.INT.RCPT.CD

# These indicators will later be used to evaluate tourism activity and tourism value within the Market Attractiveness Framework.

## KPI Selection: Economic Indicators

### Business Question
Economic indicators evaluate both current market size and future growth potential. We need to distinguish between:
- **GDP Growth** — is the economy accelerating? (forward-looking)
- **GDP Value** — how large is the economy today? (current market size)

In [28]:
# Economic indicators are included in the business case to evaluate both current market size and future growth potential.
# Several GDP-related indicators are available in the World Bank catalog. However, not all GDP measures capture the same business concept.
# Before selecting specific KPIs, an initial exploration of GDP-related indicators was conducted to understand the available options and identify the metrics most relevant to a travel market expansion strategy.

In [29]:
# Search the indicator catalog for GDP-related indicators.

gdp_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains(
        "GDP",
        case=False,
        na=False
    )
]

gdp_indicators_df[["id", "name"]].head(50)

,id,name
688,6.0.GDP_current,GDP (current $)
689,6.0.GDP_growth,GDP growth (annual %)
690,6.0.GDP_usd,GDP (constant 2005 $)
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 internation..."
2096,BG.GSR.NFSV.GD.ZS,Trade in services (% of GDP)
2097,BG.KAC.FNEI.GD.PP.ZS,"Gross private capital flows (% of GDP, PPP)"
2098,BG.KAC.FNEI.GD.ZS,Gross private capital flows (% of GDP)
2099,BG.KLT.DINV.GD.PP.ZS,"Gross foreign direct investment (% of GDP, PPP)"
2100,BG.KLT.DINV.GD.ZS,Gross foreign direct investment (% of GDP)
2401,BI.WAG.TOTL.GD.ZS,Wage bill as a percentage of GDP


In [30]:
gdp_indicators_df.nunique()

id                    588
name                  566
unit                    0
source                 15
sourceNote            195
sourceOrganization     72
topics                 23
dtype: int64

In [31]:
gdp_indicators_df["name"].value_counts()

name
Foreign direct investment, net inflows (% of GDP)                                                   3
Central government debt, total (% of GDP)                                                           3
Tax revenue (% of GDP)                                                                              3
GDP growth (annual %)                                                                               3
Inflation, GDP deflator (annual %)                                                                  2
                                                                                                   ..
Gross PSD, Nonfinancial Public Corp., Short-term, Currency and deposits, Nominal Value, % of GDP    1
Gross PSD, General Gov., Short-term, Currency and deposits, Nominal Value, % of GDP                 1
Gross PSD, Financial Public Corp., Short-term, Currency and deposits, Nominal Value, % of GDP       1
Gross PSD, Central Gov., Short-term, Currency and deposits, Nominal Value, % 

In [32]:
# The search returns a broad range of GDP-related indicators.
# Since the project aims to evaluate both market size and future growth opportunities, the analysis will focus on three economic dimensions:

# 1. GDP Growth
# 2. GDP (Current US$)

# Each dimension addresses a different business question.

### GDP Growth

In [33]:
# Search the indicator catalog for GDP growth-related indicators.

gdp_growth_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains(
        "GDP growth",
        case=False,
        na=False
    )
]

gdp_growth_indicators_df[["id", "name"]]

,id,name
689,6.0.GDP_growth,GDP growth (annual %)
15917,NV.AGR.TOTL.ZG,Real agricultural GDP growth rates (%)
16210,NY.GDP.MKTP.KD.ZG,GDP growth (annual %)
16213,NY.GDP.MKTP.KN.87.ZG,GDP growth (annual %)
16327,NYGDPMKTPKDZ,"GDP growth, constant (average 2010-19 prices a..."


In [34]:
# Several GDP growth indicators are available within the World Bank catalog.
# Some indicators measure growth across the entire economy, while others focus on specific sectors or use alternative methodologies.
# The next step is to compare the available candidates and determine which indicator best aligns with the business objective.

In [35]:
gdp_growth_candidates = gdp_growth_indicators_df[
    [
        "id",
        "name",
        "sourceNote",
        "sourceOrganization"
    ]
]

gdp_growth_candidates

,id,name,sourceNote,sourceOrganization
689,6.0.GDP_growth,GDP growth (annual %),Annual percentage growth rate of GDP at market...,World Development Indicators (World Bank)
15917,NV.AGR.TOTL.ZG,Real agricultural GDP growth rates (%),This is the annual rate of growth of agricultu...,World Bank country economists.
16210,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),Gross domestic product is the total income ear...,"Country official statistics, National Statisti..."
16213,NY.GDP.MKTP.KN.87.ZG,GDP growth (annual %),NaN,NaN
16327,NYGDPMKTPKDZ,"GDP growth, constant (average 2010-19 prices a...",GDP is the sum of gross value added by all res...,The World Bank


In [36]:
# The candidate indicators include both broad economic measures and sector-specific measures.

# For example:
# NV.AGR.TOTL.ZG measures agricultural GDP growth only and therefore does not represent overall economic performance.
# NY.GDP.MKTP.KD.ZG measures total economy GDP growth and is the standard indicator for comparing economic momentum across countries.

In [37]:
gdp_growth_candidates["name"].value_counts()

name
GDP growth (annual %)                                               3
Real agricultural GDP growth rates (%)                              1
GDP growth, constant (average 2010-19 prices and exchange rates)    1
Name: count, dtype: int64

In [38]:
# Several indicators share the same description: "GDP growth (annual %)".
# This suggests that multiple versions of the indicator exist within the catalog.
# Additional metadata will therefore be reviewed to select the most appropriate version.

In [39]:
gdp_growth_candidates[
    gdp_growth_candidates["name"] == "GDP growth (annual %)"
]

,id,name,sourceNote,sourceOrganization
689,6.0.GDP_growth,GDP growth (annual %),Annual percentage growth rate of GDP at market...,World Development Indicators (World Bank)
16210,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),Gross domestic product is the total income ear...,"Country official statistics, National Statisti..."
16213,NY.GDP.MKTP.KN.87.ZG,GDP growth (annual %),NaN,NaN


In [40]:
# Among the indicators labelled "GDP growth (annual %)", some provide source information while others contain limited metadata.
# Indicators with stronger documentation and standard World Bank naming conventions are preferred for reliability.

In [41]:
selected_gdp_growth_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "NY.GDP.MKTP.KD.ZG"
]

selected_gdp_growth_indicator[
    [
        "id",
        "name",
        "sourceNote",
    ]
]

,id,name,sourceNote
16210,NY.GDP.MKTP.KD.ZG,GDP growth (annual %),Gross domestic product is the total income ear...


In [42]:
# 2.1. GDP Growth: NY.GDP.MKTP.KD.ZG is selected as the project's economic growth KPI.

# Selection rationale:
    # Measures GDP growth for the overall economy.
    # Expressed as an annual percentage, enabling direct comparison across countries.
    # Adjusted for inflation (constant dollars), making comparisons across years more meaningful.
    # Standard World Bank measure used globally for economic momentum analysis.

### GDP Value

In [43]:
gdp_value_candidates = gdp_indicators_df[
    gdp_indicators_df["name"].str.contains(
        "GDP",
        case=False,
        na=False
    )
]

gdp_value_candidates[
    ["id", "name"]
].head(50)

,id,name
688,6.0.GDP_current,GDP (current $)
689,6.0.GDP_growth,GDP growth (annual %)
690,6.0.GDP_usd,GDP (constant 2005 $)
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 internation..."
2096,BG.GSR.NFSV.GD.ZS,Trade in services (% of GDP)
2097,BG.KAC.FNEI.GD.PP.ZS,"Gross private capital flows (% of GDP, PPP)"
2098,BG.KAC.FNEI.GD.ZS,Gross private capital flows (% of GDP)
2099,BG.KLT.DINV.GD.PP.ZS,"Gross foreign direct investment (% of GDP, PPP)"
2100,BG.KLT.DINV.GD.ZS,Gross foreign direct investment (% of GDP)
2401,BI.WAG.TOTL.GD.ZS,Wage bill as a percentage of GDP


In [44]:
# The GDP indicator catalog contains multiple measures of economic performance.
# Some indicators measure total economic output, while others focus on output per person or adjust values for inflation.

In [45]:
gdp_market_size_candidates = gdp_indicators_df[
    gdp_indicators_df["id"].isin([
        "NY.GDP.MKTP.CD",
        "NY.GDP.MKTP.KD",
        "NY.GDP.PCAP.CD",
        "NY.GDP.PCAP.KD"
    ])
]

gdp_market_size_candidates[["id", "name", "sourceNote"]]

,id,name,sourceNote
16202,NY.GDP.MKTP.CD,GDP (current US$),Gross domestic product is the total income ear...
16208,NY.GDP.MKTP.KD,GDP (constant 2015 US$),Gross domestic product is the total income ear...
16221,NY.GDP.PCAP.CD,GDP per capita (current US$),Gross domestic product is the total income ear...
16223,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),Gross domestic product is the total income ear...


In [46]:
# Four candidate indicators were reviewed.
# NY.GDP.MKTP.CD and NY.GDP.MKTP.KD measure total GDP and therefore represent overall economic market size.
# NY.GDP.PCAP.CD and NY.GDP.PCAP.KD measure GDP per person and represent consumer wealth.
# Current US$ versions were preferred over constant versions for comparability at a single point in time.

In [47]:
selected_gdp_value_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "NY.GDP.MKTP.CD"
]

selected_gdp_value_indicator[
    [
        "id",
        "name",
        "sourceNote",
    ]
]

,id,name,sourceNote
16202,NY.GDP.MKTP.CD,GDP (current US$),Gross domestic product is the total income ear...


In [48]:
# 2.2. GDP Value: NY.GDP.MKTP.CD (GDP, Current US$) was selected as the project's measure of economic market size.

# Selection rationale:
    # Measures total economic output.
    # Represents overall market size in current US dollars.
    # Used to identify and rank the Top 30 countries by GDP for the project scope.

## KPI Selection: Digital Readiness

### Business Question
Online travel platforms depend on internet access to reach customers. A country may have strong economic indicators but if its population lacks internet access, it cannot effectively be targeted through digital marketing channels.

**Why it matters:** Higher internet penetration = larger addressable online audience for a travel platform.

In [49]:
# Digital readiness is a critical component of the business case because online travel platforms rely on internet access to reach customers.
# A country may have strong tourism activity and economic growth, but without internet access, digital marketing investment would be ineffective.

In [50]:
# Searching the catalog for digital readiness insights - internet related indicators
# We filter rows where the indicator name contains the word 'internet'
# case=False - Makes it case insensitive. Ex: internet vs INTERNET
# na=False - This way, if a name is null, it will not throw an error.


internet_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains("internet", case=False, na=False)
]

print("Number of internet-related indicators found:", len(internet_indicators_df))
print()
internet_indicators_df[["id", "name"]].head(50)

Number of internet-related indicators found: 280



,id,name
114,2.0.cov.Int,Coverage: Internet
139,2.0.hoi.Int,HOI: Internet
3531,con26d,Daily internet use (% age 15+)
3532,con26d.1,"Daily internet use, women (% age 15+)"
3533,con26d.10,"Daily internet use, urban (% age 15+)"
3534,con26d.11,"Daily internet use, out of laborforce (% age 15+)"
3535,con26d.12,"Daily internet use, in laborforce (% age 15+)"
3536,con26d.2,"Daily internet use, men (% age 15+)"
3537,con26d.3,"Daily internet use, young (% ages 15-24)"
3538,con26d.4,"Daily internet use, older (% age 25+)"


In [51]:
# The search returns a variety of internet-related indicators, including measures of internet usage, broadband subscriptions, digital infrastructure, and connectivity.
# For this project, the focus is on population-level internet usage because it represents the total addressable digital audience.

internet_candidates = indicator_catalog_df[
    (
        indicator_catalog_df["name"].str.contains(
            "internet",
            case=False,
            na=False
        )
    )
    &
    (
        indicator_catalog_df["name"].str.contains(
            "population",
            case=False,
            na=False
        )
    )
]

internet_candidates[["id", "name"]]

,id,name
13530,IT.NET.USER.FE.ZS,"Individuals using the Internet, female (% of f..."
13531,IT.NET.USER.MA.ZS,"Individuals using the Internet, male (% of mal..."
13534,IT.NET.USER.ZS,Individuals using the Internet (% of population)


In [52]:
# To measure digital readiness, preference was given to indicators expressed as a percentage of the population.
# Percentage-based indicators allow meaningful comparison between countries of different sizes.

In [53]:
# We chose IT.NET.USER.ZS because:
    # Measures internet adoption at the population level.
    # Expressed as a percentage — comparable across countries of any size.
    # Standard global measure for digital readiness used by international organizations.

selected_digital_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "IT.NET.USER.ZS"
]
print("Confirmed indicator:")
print(selected_digital_indicator[["id", "name"]])

Confirmed indicator:
                   id                                              name
13534  IT.NET.USER.ZS  Individuals using the Internet (% of population)


## KPI Selection: Price Competitiveness

### Business Question
Travelers choose destinations partly based on affordability. Countries that offer better value for money attract more travelers and represent stronger growth opportunities for a targeting volume.

**How to read PPP:** The PPP conversion factor tells us how many local currency units are needed to buy the same goods that 1 US dollar buys in the US. A lower value = cheaper country = more price competitive.

In [54]:
# Price competitiveness is included in the business case to evaluate the relative affordability of different travel markets.
# For travelers, destination attractiveness is influenced not only by tourism infrastructure but also by the relative cost of visiting.

In [55]:
# Searching the catalog for PPP-related indicators
# PPP = Purchasing Power Parity
# We filter rows where the indicator name contains the word 'PPP'

ppp_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains("PPP", case=False, na=False)
]

print("Number of PPP-related indicators found:", len(ppp_indicators_df))
print()
ppp_indicators_df[["id", "name"]].head(20)

Number of PPP-related indicators found: 182



,id,name
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 internation..."
694,6.1_PRIMARY.ENERGY.INTENSITY,Energy intensity level of primary energy (MJ/$...
2097,BG.KAC.FNEI.GD.PP.ZS,"Gross private capital flows (% of GDP, PPP)"
2099,BG.KLT.DINV.GD.PP.ZS,"Gross foreign direct investment (% of GDP, PPP)"
2822,CC.EG.INTS.KW,Energy intensity of the economy (kWh per 2011$...
3202,CoCA_PPP,Cost of an energy sufficient diet in PPP dollars
3203,CoCA_PPP,Cost of an energy sufficient diet in PPP dollars
3208,CoHD_asf_PPP,Cost of animal source foods in PPP dollars
3209,CoHD_asf_PPP,Cost of animal source foods in PPP dollars
3218,CoHD_f_PPP,Cost of fruits in PPP dollars


In [56]:
# Purchasing Power Parity (PPP) indicators adjust for differences in local price levels across countries.
# PPP measures are commonly used in international economic comparisons because they allow more accurate cross-country price level comparisons.

In [57]:
# Narrowing down to find the conversion factor specifically according to the KPIs we set.
# The broad search returned many PPP indicators
# We search more specifically for 'conversion factor' to find the right one

ppp_conversion_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains("conversion factor", case=False, na=False)
]

print("Number of conversion factor indicators found:", len(ppp_conversion_df))
print()
ppp_conversion_df[["id", "name"]]

Number of conversion factor indicators found: 7



,id,name
7690,EA.NUS.FCRF,"Conversion Factor (Annual average, local per US$)"
16404,PA.NUS.ATLS,DEC alternative conversion factor (LCU per US$)
16408,PA.NUS.PPP,"PPP conversion factor, GDP (LCU per internatio..."
16409,PA.NUS.PPP.05,"2005 PPP conversion factor, GDP (LCU per inter..."
16410,PA.NUS.PPPC.RF,Price level ratio of PPP conversion factor (GD...
16412,PA.NUS.PRVT.PP,"PPP conversion factor, households and NPISHs F..."
16413,PA.NUS.PRVT.PP.05,"2005 PPP conversion factor, private consumptio..."


In [58]:
# PPP conversion factors estimate how much local currency is required to purchase the same basket of goods and services across countries.
# These indicators are particularly relevant for comparing relative price levels across travel markets.

In [59]:
# Several PPP conversion factor indicators are available.
# Some focus on overall economic activity, while others focus on specific categories such as private consumption.
# Because the project aims to evaluate overall country price levels rather than specific spending categories, the GDP-based conversion factor was selected.

In [60]:
# From the search results we identified: PA.NUS.PPP
# LCU = Local Currency Units
# We chose this because:
# - It measures how many local currency units are needed to buy what $1 buys in the US
# - Lower value = cheaper country = more price competitive for international travelers

confirmed_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "PA.NUS.PPP"
]

print("Confirmed indicator:")
print(confirmed_indicator[["id", "name"]])

Confirmed indicator:
               id                                               name
16408  PA.NUS.PPP  PPP conversion factor, GDP (LCU per internatio...


In [61]:
# PA.NUS.PPP (PPP Conversion Factor, GDP) was selected as the project's measure of price competitiveness.

# Selection rationale:
    # Measures relative price levels across countries.
    # Lower value = cheaper country = more price competitive destination for travelers.
    # GDP-based version preferred because it reflects overall economy price levels rather than a specific spending category.
    # IMPORTANT FOR SCORING: Unlike other indicators where higher = better, for PPP lower = better.
    # This indicator must be INVERTED when calculating the Market Attractiveness Score.

## KPI Selection: Supporting Indicators

These indicators are not included in the core Market Attractiveness Score but will support EDA analysis and provide additional context for the business recommendation.

### GDP per Capita

In [62]:
# While total GDP measures the size of an economy, GDP per capita measures average economic output per person.
# This indicator was evaluated because it may provide additional insight into consumer purchasing power and travel spending potential.

In [63]:
gdp_per_capita_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains(
        "GDP per capita",
        case=False,
        na=False
    )
]

gdp_per_capita_indicators_df[["id", "name"]]

,id,name
691,6.0.GDPpc_constant,"GDP per capita, PPP (constant 2011 internation..."
2944,CC.GHG.MEMG.GC,Macro drivers of GHG emissions growth in the p...
8172,FB.DPT.INSU.PC.ZS,Deposit insurance coverage (% of GDP per capita)
15907,NV.AGR.PCAP.KD.ZG,Real agricultural GDP per capita growth rate (%)
16221,NY.GDP.PCAP.CD,GDP per capita (current US$)
16222,NY.GDP.PCAP.CN,GDP per capita (current LCU)
16223,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$)
16224,NY.GDP.PCAP.KD.ZG,GDP per capita growth (annual %)
16225,NY.GDP.PCAP.KN,GDP per capita (constant LCU)
16226,NY.GDP.PCAP.PP.CD,"GDP per capita, PPP (current international $)"


In [64]:
# Several GDP per capita indicators are available.
# These indicators differ primarily in whether values are reported in current prices or adjusted for inflation.

In [65]:
gdp_per_capita_candidates = gdp_per_capita_indicators_df[
    gdp_per_capita_indicators_df["id"].isin([
        "NY.GDP.PCAP.CD",
        "NY.GDP.PCAP.KD"
    ])
]

gdp_per_capita_candidates[
    [
        "id",
        "name",
        "sourceNote"
    ]
]

,id,name,sourceNote
16221,NY.GDP.PCAP.CD,GDP per capita (current US$),Gross domestic product is the total income ear...
16223,NY.GDP.PCAP.KD,GDP per capita (constant 2015 US$),Gross domestic product is the total income ear...


In [66]:
# Two primary GDP per capita indicators were identified:
    # NY.GDP.PCAP.CD reports GDP per capita in current US dollars.
    # NY.GDP.PCAP.KD reports GDP per capita in constant US dollars after adjusting for inflation.

In [67]:
selected_gdp_per_capita_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "NY.GDP.PCAP.CD"
]

selected_gdp_per_capita_indicator[
    [
        "id",
        "name",
        "sourceNote"
    ]
]

,id,name,sourceNote
16221,NY.GDP.PCAP.CD,GDP per capita (current US$),Gross domestic product is the total income ear...


In [68]:
# NY.GDP.PCAP.CD (GDP per Capita, Current US$) was selected as the supporting measure of consumer wealth.
# Selection rationale:
    # Measures average economic output per person.
    # Provides insight into the purchasing power of consumers in each market.
    # Used in EDA to provide context alongside GDP growth and tourism indicators.

### Population

In [69]:
# Population was evaluated as a supporting market size indicator.
# While GDP measures economic output, population measures the size of the potential customer base.
# This indicator may provide additional context when comparing large and small economies.

In [70]:
# Search the indicator catalog for population-related metrics.

population_indicators_df = indicator_catalog_df[
    (
        indicator_catalog_df["name"].str.contains(
            "population",
            case=False,
            na=False
        )
    )
]

population_indicators_df[["id", "name"]].head(20)

,id,name
24,1.1_ACCESS.ELECTRICITY.TOT,Access to electricity (% of total population)
39,1.2_ACCESS.ELECTRICITY.RURAL,Access to electricity (% of rural population)
40,1.3_ACCESS.ELECTRICITY.URBAN,Access to electricity (% of urban population)
161,2.1_ACCESS.CFT.TOT,Access to Clean Fuels and Technologies for coo...
1552,allsa.cov_pop,Coverage of social safety net programs (% of p...
1555,allsi.cov_pop,Coverage of social insurance programs (% of po...
1558,allsp.cov_pop,Coverage of social protection and labor progra...
1670,BAR.NOED.1519.FE.ZS,Barro-Lee: Percentage of female population age...
1671,BAR.NOED.1519.ZS,Barro-Lee: Percentage of population age 15-19 ...
1672,BAR.NOED.15UP.FE.ZS,Barro-Lee: Percentage of female population age...


In [71]:
# The search returns several population-related indicators, including total population, urban population counts, rural population counts, and age-specific population measures.
# For this project, the total population indicator is most relevant.

In [72]:
population_candidates = population_indicators_df[
    population_indicators_df["name"].str.contains(
        "total",
        case=False,
        na=False
    )
]

population_candidates[
    ["id", "name"]
].head(20)

,id,name
24,1.1_ACCESS.ELECTRICITY.TOT,Access to electricity (% of total population)
161,2.1_ACCESS.CFT.TOT,Access to Clean Fuels and Technologies for coo...
1700,BAR.POP.1519,"Barro-Lee: Population in thousands, age 15-19,..."
1702,BAR.POP.15UP,"Barro-Lee: Population in thousands, age 15+, t..."
1704,BAR.POP.2024,"Barro-Lee: Population in thousands, age 20-24,..."
1706,BAR.POP.2529,"Barro-Lee: Population in thousands, age 25-29,..."
1708,BAR.POP.25UP,"Barro-Lee: Population in thousands, age 25+, t..."
1710,BAR.POP.3034,"Barro-Lee: Population in thousands, age 30-34,..."
1712,BAR.POP.3539,"Barro-Lee: Population in thousands, age 35-39,..."
1714,BAR.POP.4044,"Barro-Lee: Population in thousands, age 40-44,..."


In [73]:
# Indicators containing "total" were reviewed because they represent complete population counts rather than specific population segments.
# A total population measure is more appropriate for cross-country market size comparison.

In [74]:
selected_population_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "SP.POP.TOTL"
]

selected_population_indicator[
    [
        "id",
        "name",
        "sourceNote",
    ]
]

,id,name,sourceNote
26888,SP.POP.TOTL,"Population, total",Total population is based on the de facto defi...


In [75]:
# SP.POP.TOTL (Population, Total) was selected as the project's supporting population indicator.

# Selection rationale:
    # Measures the total number of residents in a country.
    # Provides a direct measure of potential market size.
    # Used to provide context in EDA alongside GDP and tourism indicators.

### Urban Population %

In [76]:
# Urban Population % was evaluated as a supporting indicator for market accessibility.
# While total population measures the size of the potential customer base, urban population share provides context on how accessible digital travel platforms are likely to be.

In [77]:
# Search the indicator catalog for urban population-related metrics.

urban_population_indicators_df = indicator_catalog_df[
    (
        indicator_catalog_df["name"].str.contains(
            "urban",
            case=False,
            na=False
        )
    )
    &
    (
        indicator_catalog_df["name"].str.contains(
            "population",
            case=False,
            na=False
        )
    )
]

urban_population_indicators_df[["id", "name"]].head(20)

,id,name
40,1.3_ACCESS.ELECTRICITY.URBAN,Access to electricity (% of urban population)
7764,EG.CFT.ACCS.UR.ZS,Access to clean fuels and technologies for coo...
7769,EG.ELC.ACCS.UR.ZS,"Access to electricity, urban (% of urban popul..."
7797,EG.NSF.ACCS.UR.ZS,"Access to non-solid fuel, urban (% of urban po..."
7867,empl_al_alot_dfcl_urb,Employment to population ratio (% of persons l...
7882,empl_any_dfcl_urb,Employment to population ratio (% of persons l...
7891,empl_none_dfcl_urb,Employment to population ratio (% of persons l...
7900,empl_nosome_dfcl_urb,Employment to population ratio (% of persons l...
7909,empl_some_dfcl_urb,Employment to population ratio (% of persons l...
8049,EN.POP.EL5M.UR.ZS,Urban population living in areas where elevati...


In [78]:
# The initial search returns urban population indicators covering both total urban population counts and urban population shares.
# For this supporting KPI, the objective is to compare urbanization levels across countries — therefore a percentage-based indicator is preferred.

In [79]:
# Narrow the list to percentage-based urban population indicators.
# Percentage-based indicators allow comparison across countries of different sizes.

urban_population_percentage_candidates = urban_population_indicators_df[
    urban_population_indicators_df["name"].str.contains(
        "%",
        case=False,
        na=False
    )
]

urban_population_percentage_candidates[["id", "name"]]

,id,name
40,1.3_ACCESS.ELECTRICITY.URBAN,Access to electricity (% of urban population)
7764,EG.CFT.ACCS.UR.ZS,Access to clean fuels and technologies for coo...
7769,EG.ELC.ACCS.UR.ZS,"Access to electricity, urban (% of urban popul..."
7797,EG.NSF.ACCS.UR.ZS,"Access to non-solid fuel, urban (% of urban po..."
7867,empl_al_alot_dfcl_urb,Employment to population ratio (% of persons l...
...,...,...
28048,UIS.LR.AG25T64.URB.M,"Literacy rate, population 25-64 years, urban, ..."
28060,UIS.LR.AG65T99.URB,"Elderly literacy rate, population 65+ years, u..."
28061,UIS.LR.AG65T99.URB.F,"Elderly literacy rate, population 65+ years, u..."
28063,UIS.LR.AG65T99.URB.M,"Elderly literacy rate, population 65+ years, u..."


In [80]:
# After narrowing the list to percentage-based indicators, the selected KPI should represent urban population as a share of total population.

In [81]:
# Further narrow the percentage-based indicators to those related to total population.
# This removes indicators focused on age groups, gender groups, or other population segments.

urban_population_total_percentage_candidates = urban_population_percentage_candidates[
    urban_population_percentage_candidates["name"].str.contains(
        "total",
        case=False,
        na=False
    )
]

urban_population_total_percentage_candidates[["id", "name"]]

,id,name
8049,EN.POP.EL5M.UR.ZS,Urban population living in areas where elevati...
8062,EN.URB.MCTY.TL.ZS,Population in urban agglomerations of more tha...
14141,JI.POP.URBN.ZS,"Urban population, total (% of total population)"
26937,SP.URB.MCTY.UR.ZS,Population in urban agglomerations > 1 million...
26939,SP.URB.TOTL.FE.ZS,"Urban population, female (% of total)"
26940,SP.URB.TOTL.IN.ZS,Urban population (% of total population)
26941,SP.URB.TOTL.MA.ZS,"Urban population, male (% of total)"
26942,SP.URB.TOTL.ZS,Percentage of Population in Urban Areas (in % ...
26943,SP.URB.TOTL.ZS.SUS,Percentage of Population in Urban Areas (in % ...


In [82]:
# Narrow the list to indicators specifically measuring urban population as a share of total population.

urban_population_share_candidates = urban_population_total_percentage_candidates[
    urban_population_total_percentage_candidates["name"].str.contains(
        "of total",
        case=False,
        na=False
    )
]

urban_population_share_candidates[["id", "name"]]

,id,name
8049,EN.POP.EL5M.UR.ZS,Urban population living in areas where elevati...
8062,EN.URB.MCTY.TL.ZS,Population in urban agglomerations of more tha...
14141,JI.POP.URBN.ZS,"Urban population, total (% of total population)"
26937,SP.URB.MCTY.UR.ZS,Population in urban agglomerations > 1 million...
26939,SP.URB.TOTL.FE.ZS,"Urban population, female (% of total)"
26940,SP.URB.TOTL.IN.ZS,Urban population (% of total population)
26941,SP.URB.TOTL.MA.ZS,"Urban population, male (% of total)"
26942,SP.URB.TOTL.ZS,Percentage of Population in Urban Areas (in % ...
26943,SP.URB.TOTL.ZS.SUS,Percentage of Population in Urban Areas (in % ...


In [83]:
selected_urban_population_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "SP.URB.TOTL.IN.ZS"
]

selected_urban_population_indicator[["id", "name", "sourceNote", "sourceOrganization"]]

,id,name,sourceNote,sourceOrganization
26940,SP.URB.TOTL.IN.ZS,Urban population (% of total population),Urban population refers to people living in ur...,"World Urbanization Prospects, United Nations (..."


In [84]:
# SP.URB.TOTL.IN.ZS (Urban Population % of Total Population) was selected as the supporting urbanization indicator.
# This indicator measures the share of a country's population living in urban areas.

### Mobile Subscriptions

In [85]:
# Mobile connectivity was evaluated as a supporting indicator for digital readiness.
# While internet penetration measures online access, mobile subscriptions may provide additional insight into how consumers access travel platforms.

In [86]:
# Search the indicator catalog for mobile-related metrics.

mobile_indicators_df = indicator_catalog_df[
    indicator_catalog_df["name"].str.contains(
        "mobile",
        case=False,
        na=False
    )
]

mobile_indicators_df[["id", "name"]].head(20)

,id,name
111,2.0.cov.Cel,Coverage: Mobile Phone
136,2.0.hoi.Cel,HOI: Mobile Phone
3266,con1,Own a mobile phone (% age 15+)
3267,con1.1,"Own a mobile phone, women (% age 15+)"
3268,con1.10,"Own a mobile phone, urban (% age 15+)"
3269,con1.11,"Own a mobile phone, out of laborforce (% age 15+)"
3270,con1.12,"Own a mobile phone, in laborforce (% age 15+)"
3271,con1.2,"Own a mobile phone, men (% age 15+)"
3272,con1.3,"Own a mobile phone, young (% ages 15-24)"
3273,con1.4,"Own a mobile phone, older (% age 25+)"


In [87]:
# The initial search returns a broad range of mobile-related indicators, including subscriptions, coverage, network infrastructure, and usage metrics.
# The objective is to identify a measure that reflects actual consumer mobile adoption.

In [88]:
# Focus on indicators related to subscriptions.

mobile_subscription_candidates = mobile_indicators_df[
    mobile_indicators_df["name"].str.contains(
        "subscription",
        case=False,
        na=False
    )
]

mobile_subscription_candidates[["id", "name"]].head(20)

,id,name
13453,IT.CEL.SETS,Mobile cellular subscriptions
13454,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people)
13461,IT.CELL.MSUB.CD,Mobile cellular monthly subscription (current ...
13462,IT.CELL.MSUB.CN,Mobile cellular monthly subscription (current ...
13552,IT.TEL.TOTL.EM,Fixed line and mobile cellular subscriptions p...
13553,IT.TEL.TOTL.P2,Fixed line and mobile cellular subscriptions (...


In [89]:
# Subscription-based indicators are more relevant than infrastructure indicators because they reflect actual adoption and usage by consumers.

In [90]:
# Focus on indicators normalized by population size.

mobile_per_capita_candidates = mobile_subscription_candidates[
    mobile_subscription_candidates["name"].str.contains(
        "100 people|per 100",
        case=False,
        na=False
    )
]

mobile_per_capita_candidates[["id", "name"]]

,id,name
13454,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people)
13553,IT.TEL.TOTL.P2,Fixed line and mobile cellular subscriptions (...


In [91]:
selected_mobile_subscription_indicator = indicator_catalog_df[
    indicator_catalog_df["id"] == "IT.CEL.SETS.P2"
]

selected_mobile_subscription_indicator[["id", "name", "sourceNote", "sourceOrganization"]]

,id,name,sourceNote,sourceOrganization
13454,IT.CEL.SETS.P2,Mobile cellular subscriptions (per 100 people),Mobile cellular telephone subscriptions are su...,World Telecommunication/ICT Indicators Databas...


In [92]:
# IT.CEL.SETS.P2 (Mobile Cellular Subscriptions per 100 People) was selected as the supporting mobile connectivity indicator.
# Selection rationale:
    # Measures mobile adoption rather than infrastructure.
    # Expressed per 100 people — comparable across countries of different sizes.
    # Provides supporting context on digital accessibility alongside internet penetration.

## Final Dataframe for EDA

All selected indicators are compiled into a single reference DataFrame and saved to disk.

In [93]:
selected_world_bank_kpis_df = pd.DataFrame({
    "business_dimension": [
        "Tourism Activity",
        "Tourism Value",
        "Economic Growth",
        "Economic Market Size",
        "Digital Readiness",
        "Price Competitiveness",
        "Supporting — Consumer Wealth",
        "Supporting — Market Size",
        "Supporting — Urbanization",
        "Supporting — Mobile Connectivity"
    ],
    "indicator_code": [
        "ST.INT.ARVL",
        "ST.INT.RCPT.CD",
        "NY.GDP.MKTP.KD.ZG",
        "NY.GDP.MKTP.CD",
        "IT.NET.USER.ZS",
        "PA.NUS.PPP",
        "NY.GDP.PCAP.CD",
        "SP.POP.TOTL",
        "SP.URB.TOTL.IN.ZS",
        "IT.CEL.SETS.P2"
    ],
    "role": [
        "Core scoring indicator",
        "Core scoring indicator",
        "Core scoring indicator",
        "Supporting — EDA only",
        "Core scoring indicator",
        "Core scoring indicator",
        "Supporting — EDA only",
        "Supporting — EDA only",
        "Supporting — EDA only",
        "Supporting — EDA only"
    ]
})

selected_world_bank_kpis_df

,business_dimension,indicator_code,role
0,Tourism Activity,ST.INT.ARVL,Core scoring indicator
1,Tourism Value,ST.INT.RCPT.CD,Core scoring indicator
2,Economic Growth,NY.GDP.MKTP.KD.ZG,Core scoring indicator
3,Economic Market Size,NY.GDP.MKTP.CD,Supporting — EDA only
4,Digital Readiness,IT.NET.USER.ZS,Core scoring indicator
5,Price Competitiveness,PA.NUS.PPP,Core scoring indicator
6,Supporting — Consumer Wealth,NY.GDP.PCAP.CD,Supporting — EDA only
7,Supporting — Market Size,SP.POP.TOTL,Supporting — EDA only
8,Supporting — Urbanization,SP.URB.TOTL.IN.ZS,Supporting — EDA only
9,Supporting — Mobile Connectivity,IT.CEL.SETS.P2,Supporting — EDA only


In [94]:
selected_world_bank_kpis_df.to_csv(
    "selected_world_bank_kpis.csv",
    index=False
)

## Country List

### Country Selection Steps
The analysis scope is defined as the **Top 30 countries by GDP**. This choice was made because:
- The largest economies represent the biggest potential travel markets
- High GDP countries tend to have more complete World Bank data, reducing null values
- For a travel company deciding where to invest marketing budget, the world's largest economies are the logical starting point

**Note on Russia and Turkey:** Both countries are included in the Top 30 by GDP but carry geopolitical risk flags. The team acknowledged this as an assumption — results involving these countries should be interpreted with caution and may be excluded from the final recommendation depending on the client's risk appetite.

In [95]:
!pip install pycountry

In [96]:
import requests
import pandas as pd
import pycountry

In [97]:
# The World Bank API includes both countries and aggregated regions.
# Examples of aggregations are: `World`, `High income`, `European Union`, `OECD members`
# These are not individual countries, so they should not be included in the country ranking.
# To remove them, we will use `pycountry` and keep only official ISO 3-letter country codes.

In [98]:
def is_valid_country(country_code):
    """
    Returns True if the country code is an official ISO alpha-3 country code.
    Returns False for World Bank aggregations such as World, High income, or European Union.
    """
    try:
        return pycountry.countries.get(alpha_3=country_code) is not None
    except:
        return False

In [99]:
# 1. GDP VALUE: This will help us identify the Top 30 largest economies in 2024.

In [100]:
gdp_indicator_code = "NY.GDP.MKTP.CD"

url = f"https://api.worldbank.org/v2/country/all/indicator/{gdp_indicator_code}"

params = {
    "format": "json",
    "date": "2019:2024",
    "per_page": 20000
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [101]:
gdp_raw_data = response.json()

gdp_raw_df = pd.DataFrame(gdp_raw_data[1])

gdp_raw_df.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,1.242694e+12,,,0
1,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,1.179359e+12,,,0
2,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,1.228968e+12,,,0
3,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,1.114145e+12,,,0
4,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,9.386076e+11,,,0


In [102]:
# The raw API output includes nested country information and several columns that are not needed for this analysis.
# Next, we will keep only the columns needed for country-level analysis.

In [103]:
gdp_df = gdp_raw_df[["country", "countryiso3code", "date", "value"]].copy()

gdp_df.head()

,country,countryiso3code,date,value
0,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,1.242694e+12
1,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,1.179359e+12
2,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,1.228968e+12
3,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,1.114145e+12
4,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,9.386076e+11


In [104]:
gdp_df["country"] = gdp_df["country"].apply(lambda x: x["value"])

gdp_df = gdp_df.rename(columns={
    "countryiso3code": "country_code",
    "date": "year",
    "value": "gdp_usd"
})

gdp_df["year"] = gdp_df["year"].astype(int)

gdp_df.head()

,country,country_code,year,gdp_usd
0,Africa Eastern and Southern,AFE,2024,1.242694e+12
1,Africa Eastern and Southern,AFE,2023,1.179359e+12
2,Africa Eastern and Southern,AFE,2022,1.228968e+12
3,Africa Eastern and Southern,AFE,2021,1.114145e+12
4,Africa Eastern and Southern,AFE,2020,9.386076e+11


In [105]:
# Now we remove World Bank aggregations and keep only valid countries.

In [106]:
gdp_df["is_valid_country"] = gdp_df["country_code"].apply(is_valid_country)

gdp_df["is_valid_country"].value_counts()

is_valid_country
True     1290
False     306
Name: count, dtype: int64

In [107]:
gdp_country_df = gdp_df[gdp_df["is_valid_country"] == True].copy()

gdp_country_df = gdp_country_df.drop(columns=["is_valid_country"])

gdp_country_df.head()

,country,country_code,year,gdp_usd
294,Afghanistan,AFG,2024,NaN
295,Afghanistan,AFG,2023,1.715223e+10
296,Afghanistan,AFG,2022,1.449724e+10
297,Afghanistan,AFG,2021,1.426000e+10
298,Afghanistan,AFG,2020,1.995593e+10


In [108]:
# Next, we remove rows where GDP is missing. Missing GDP values cannot be used for ranking countries.

In [109]:
gdp_country_df = gdp_country_df.dropna(subset=["gdp_usd"])

gdp_country_df.isna().sum()

country         0
country_code    0
year            0
gdp_usd         0
dtype: int64

In [110]:
# Now we filter the dataset to 2024 and identify the Top 30 countries by GDP.

In [111]:
gdp_2024_df = gdp_country_df[gdp_country_df["year"] == 2024].copy()

gdp_2024_df.head()

,country,country_code,year,gdp_usd
300,Albania,ALB,2024,2.704643e+10
306,Algeria,DZA,2024,2.693223e+11
318,Andorra,AND,2024,4.039842e+09
324,Angola,AGO,2024,1.009989e+11
330,Antigua and Barbuda,ATG,2024,2.207623e+09


In [112]:
top_30_gdp_2024_df = gdp_2024_df.sort_values(
    by="gdp_usd",
    ascending=False
).head(30)

top_30_gdp_2024_df

,country,country_code,year,gdp_usd
1530,United States,USA,2024,2.875096e+13
540,China,CHN,2024,1.874380e+13
732,Germany,DEU,2024,4.685593e+12
882,Japan,JPN,2024,4.027598e+12
828,India,IND,2024,3.909892e+12
1524,United Kingdom,GBR,2024,3.686033e+12
702,France,FRA,2024,3.160443e+12
870,Italy,ITA,2024,2.380825e+12
504,Canada,CAN,2024,2.243637e+12
450,Brazil,BRA,2024,2.185822e+12


In [ ]:
# Saving the Top 30 country list so it can be loaded in Notebook 3

top_30_gdp_2024_df.to_excel("final_country_list_based_on_gdp.xlsx", index=False)


final_country_list_based_on_gdp.xlsx saved successfully.


In [114]:
print("Number of countries:", len(top_30_gdp_2024_df))
print("Number of unique country codes:", top_30_gdp_2024_df["country_code"].nunique())

Number of countries: 30
Number of unique country codes: 30


In [115]:
top_30_country_codes = top_30_gdp_2024_df["country_code"].tolist()

top_30_country_codes

['USA',
 'CHN',
 'DEU',
 'JPN',
 'IND',
 'GBR',
 'FRA',
 'ITA',
 'CAN',
 'BRA',
 'RUS',
 'KOR',
 'MEX',
 'AUS',
 'ESP',
 'IDN',
 'TUR',
 'SAU',
 'NLD',
 'CHE',
 'POL',
 'BEL',
 'ARG',
 'IRL',
 'SWE',
 'ARE',
 'SGP',
 'ISR',
 'AUT',
 'THA']

In [116]:
# Defining our country list based on the team's finalised Top 30 by GDP
# Using ISO 2-letter country codes as required by the World Bank API

countries = [
    "US", "CN", "DE", "JP", "IN", "GB", "FR", "IT", "CA", "BR",
    "RU", "KR", "MX", "AU", "ES", "ID", "TR", "SA", "NL", "CH",
    "PL", "BE", "AR", "IE", "SE", "AE", "SG", "IL", "AT", "TH"
]

# Joining the list into one string separated by ;
# The World Bank API requires this format to request multiple countries in one call
# Example result: "US;CN;DE;JP..."
countries_joined = ";".join(countries)

print("Number of countries:", len(countries))
print()
print(countries_joined)

Number of countries: 30

US;CN;DE;JP;IN;GB;FR;IT;CA;BR;RU;KR;MX;AU;ES;ID;TR;SA;NL;CH;PL;BE;AR;IE;SE;AE;SG;IL;AT;TH


## Tourism Arrivals

### About This Indicator
- **Indicator:** ST.INT.ARVL — International Tourism, Number of Arrivals
- **Why:** Measures the total number of international visitors entering a country — countries with higher arrival volumes represent larger, more established tourism markets which are more attractive for a travel company's marketing investment
- **Years:** 2013–2022 (10 years for trend analysis; 2022 filtered separately for scoring)
- **Scoring note:** HIGHER tourism arrivals = HIGHER score. Standard direction — no inversion needed.

In [117]:
tourism_arrivals_indicator = "ST.INT.ARVL"

In [118]:
url = f"https://api.worldbank.org/v2/country/all/indicator/{tourism_arrivals_indicator}"

params = {
    "format": "json",
    "date": "2018:2020",
    "per_page": 20000
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [119]:
tourism_raw_data = response.json()

tourism_raw_df = pd.DataFrame(tourism_raw_data[1])

tourism_raw_df.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'ST.INT.ARVL', 'value': 'International ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,NaN,,,0
1,"{'id': 'ST.INT.ARVL', 'value': 'International ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2019,3.982670e+07,,,0
2,"{'id': 'ST.INT.ARVL', 'value': 'International ...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2018,4.118915e+07,,,0
3,"{'id': 'ST.INT.ARVL', 'value': 'International ...","{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2020,NaN,,,0
4,"{'id': 'ST.INT.ARVL', 'value': 'International ...","{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2019,NaN,,,0


In [120]:
# The raw tourism dataset contains both countries and regional aggregations.
# Cleaning up the data

tourism_df = tourism_raw_df[
    ["country", "countryiso3code", "date", "value"]
].copy()

tourism_df.head()

,country,countryiso3code,date,value
0,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,NaN
1,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2019,3.982670e+07
2,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2018,4.118915e+07
3,"{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2020,NaN
4,"{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2019,NaN


In [121]:
tourism_df["country"] = tourism_df["country"].apply(
    lambda x: x["value"]
)

tourism_df = tourism_df.rename(columns={
    "countryiso3code": "country_code",
    "date": "year",
    "value": "tourism_arrivals"
})

tourism_df["year"] = tourism_df["year"].astype(int)

tourism_df.head()

,country,country_code,year,tourism_arrivals
0,Africa Eastern and Southern,AFE,2020,NaN
1,Africa Eastern and Southern,AFE,2019,3.982670e+07
2,Africa Eastern and Southern,AFE,2018,4.118915e+07
3,Africa Western and Central,AFW,2020,NaN
4,Africa Western and Central,AFW,2019,NaN


In [122]:
tourism_df["is_valid_country"] = tourism_df["country_code"].apply(
    is_valid_country
)

tourism_country_df = tourism_df[
    tourism_df["is_valid_country"] == True
].copy()

tourism_country_df = tourism_country_df.drop(
    columns=["is_valid_country"]
)

tourism_country_df.head()

,country,country_code,year,tourism_arrivals
147,Afghanistan,AFG,2020,NaN
148,Afghanistan,AFG,2019,NaN
149,Afghanistan,AFG,2018,NaN
150,Albania,ALB,2020,2658000.0
151,Albania,ALB,2019,6406000.0


In [123]:
# Next, we keep only the Top 30 GDP countries identified earlier.
tourism_top30_df = tourism_country_df[
    tourism_country_df["country_code"].isin(top_30_country_codes)
].copy()

tourism_top30_df.head()

,country,country_code,year,tourism_arrivals
168,Argentina,ARG,2020,NaN
169,Argentina,ARG,2019,7399000.0
170,Argentina,ARG,2018,6942000.0
177,Australia,AUS,2020,1828000.0
178,Australia,AUS,2019,9466000.0


In [124]:
print(
    "Countries represented:",
    tourism_top30_df["country_code"].nunique()
)

Countries represented: 30


In [125]:
# Since 2019 is the last full pre-pandemic year, we will use only 2019 tourism arrivals data.

tourism_2019_df = tourism_top30_df[
    tourism_top30_df["year"] == 2019
].copy()

tourism_2019_df.head()

,country,country_code,year,tourism_arrivals
169,Argentina,ARG,2019,7399000.0
178,Australia,AUS,2019,9466000.0
181,Austria,AUT,2019,31884000.0
202,Belgium,BEL,2019,9343000.0
226,Brazil,BRA,2019,6353000.0


In [126]:
tourism_2019_df = tourism_2019_df.dropna(
    subset=["tourism_arrivals"]
)

tourism_2019_df.sort_values(
    by="tourism_arrivals",
    ascending=False
)

,country,country_code,year,tourism_arrivals
352,France,FRA,2019,217877000.0
766,United States,USA,2019,165478000.0
271,China,CHN,2019,162538000.0
685,Spain,ESP,2019,126170000.0
529,Mexico,MEX,2019,97406000.0
436,Italy,ITA,2019,95399000.0
616,Poland,POL,2019,88515000.0
742,Turkiye,TUR,2019,51747000.0
763,United Kingdom,GBR,2019,40857000.0
724,Thailand,THA,2019,39916000.0


In [127]:
print(
    "Countries with tourism data:",
    tourism_2019_df["country_code"].nunique()
)

Countries with tourism data: 30


### Tourism Arrivals Coverage Check

All Top 30 GDP countries have tourism arrivals data available for 2019.

This confirms that the indicator provides complete coverage for the selected market universe and can be included in the market attractiveness framework.

In [128]:
tourism_2019_df.sort_values(
    by="tourism_arrivals",
    ascending=False
)

,country,country_code,year,tourism_arrivals
352,France,FRA,2019,217877000.0
766,United States,USA,2019,165478000.0
271,China,CHN,2019,162538000.0
685,Spain,ESP,2019,126170000.0
529,Mexico,MEX,2019,97406000.0
436,Italy,ITA,2019,95399000.0
616,Poland,POL,2019,88515000.0
742,Turkiye,TUR,2019,51747000.0
763,United Kingdom,GBR,2019,40857000.0
724,Thailand,THA,2019,39916000.0


### Initial Observation

Several countries combine both large economic size and strong tourism activity.

Tourism Arrivals adds an industry-specific perspective to the framework by capturing the attractiveness and maturity of a country's travel market.

Unlike GDP and GDP Growth, this indicator is directly related to travel demand.

## Tourism Receipts

### About This Indicator
- **Indicator:** ST.INT.RCPT.CD — International Tourism, Receipts (Current US$)
- **Why:** Measures the economic value generated by international visitors — unlike arrivals which count visitor volume, receipts capture how much tourists actually spend, making it a stronger signal of a country's tourism revenue potential for a travel company
- **Years:** 2013–2022 (10 years for trend analysis; 2022 filtered separately for scoring)
- **Scoring note:** HIGHER tourism receipts = HIGHER score. Standard direction — no inversion needed.

In [129]:
tourism_receipts_indicator = "ST.INT.RCPT.CD"

In [130]:
url = f"https://api.worldbank.org/v2/country/all/indicator/{tourism_receipts_indicator}"

params = {
    "format": "json",
    "date": "2018:2020",
    "per_page": 20000
}

response = requests.get(url, params=params)

print("Status code:", response.status_code)

Status code: 200


In [131]:
tourism_receipts_raw_data = response.json()

tourism_receipts_raw_df = pd.DataFrame(
    tourism_receipts_raw_data[1]
)

tourism_receipts_raw_df.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'ST.INT.RCPT.CD', 'value': 'Internation...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,NaN,,,0
1,"{'id': 'ST.INT.RCPT.CD', 'value': 'Internation...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2019,2.781094e+10,,,0
2,"{'id': 'ST.INT.RCPT.CD', 'value': 'Internation...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2018,2.867264e+10,,,0
3,"{'id': 'ST.INT.RCPT.CD', 'value': 'Internation...","{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2020,NaN,,,0
4,"{'id': 'ST.INT.RCPT.CD', 'value': 'Internation...","{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2019,NaN,,,0


In [132]:
# Cleaning the data

tourism_receipts_df = tourism_receipts_raw_df[
    ["country", "countryiso3code", "date", "value"]
].copy()

tourism_receipts_df.head()

,country,countryiso3code,date,value
0,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2020,NaN
1,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2019,2.781094e+10
2,"{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2018,2.867264e+10
3,"{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2020,NaN
4,"{'id': 'ZI', 'value': 'Africa Western and Cent...",AFW,2019,NaN


In [133]:
tourism_receipts_df["country"] = tourism_receipts_df[
    "country"
].apply(lambda x: x["value"])

tourism_receipts_df = tourism_receipts_df.rename(
    columns={
        "countryiso3code": "country_code",
        "date": "year",
        "value": "tourism_receipts_usd"
    }
)

tourism_receipts_df["year"] = tourism_receipts_df["year"].astype(int)

tourism_receipts_df.head()

,country,country_code,year,tourism_receipts_usd
0,Africa Eastern and Southern,AFE,2020,NaN
1,Africa Eastern and Southern,AFE,2019,2.781094e+10
2,Africa Eastern and Southern,AFE,2018,2.867264e+10
3,Africa Western and Central,AFW,2020,NaN
4,Africa Western and Central,AFW,2019,NaN


In [134]:
tourism_receipts_df["is_valid_country"] = (
    tourism_receipts_df["country_code"]
    .apply(is_valid_country)
)

tourism_receipts_country_df = tourism_receipts_df[
    tourism_receipts_df["is_valid_country"] == True
].copy()

tourism_receipts_country_df = (
    tourism_receipts_country_df
    .drop(columns=["is_valid_country"])
)

tourism_receipts_country_df.head()

,country,country_code,year,tourism_receipts_usd
147,Afghanistan,AFG,2020,7.500000e+07
148,Afghanistan,AFG,2019,8.500000e+07
149,Afghanistan,AFG,2018,5.000000e+07
150,Albania,ALB,2020,1.243000e+09
151,Albania,ALB,2019,2.458000e+09


In [135]:
tourism_receipts_top30_df = tourism_receipts_country_df[
    tourism_receipts_country_df["country_code"]
    .isin(top_30_country_codes)
].copy()

tourism_receipts_top30_df.head()

,country,country_code,year,tourism_receipts_usd
168,Argentina,ARG,2020,1.702000e+09
169,Argentina,ARG,2019,5.654000e+09
170,Argentina,ARG,2018,5.999000e+09
177,Australia,AUS,2020,2.623400e+10
178,Australia,AUS,2019,4.795300e+10


In [136]:
print(
    "Countries represented:",
    tourism_receipts_top30_df["country_code"].nunique()
)

Countries represented: 30


In [137]:
# To remain consistent with Tourism Arrivals, we use 2019 as the reference year because it represents the last full pre-pandemic year.

tourism_receipts_2019_df = tourism_receipts_top30_df[
    tourism_receipts_top30_df["year"] == 2019
].copy()

tourism_receipts_2019_df.head()

,country,country_code,year,tourism_receipts_usd
169,Argentina,ARG,2019,5.654000e+09
178,Australia,AUS,2019,4.795300e+10
181,Austria,AUT,2019,2.592400e+10
202,Belgium,BEL,2019,1.058100e+10
226,Brazil,BRA,2019,6.127000e+09


In [138]:
tourism_receipts_2019_df = (
    tourism_receipts_2019_df
    .dropna(subset=["tourism_receipts_usd"])
)

tourism_receipts_2019_df.sort_values(
    by="tourism_receipts_usd",
    ascending=False
)

,country,country_code,year,tourism_receipts_usd
766,United States,USA,2019,2.394470e+11
352,France,FRA,2019,7.077600e+10
724,Thailand,THA,2019,6.437100e+10
367,Germany,DEU,2019,5.837200e+10
436,Italy,ITA,2019,5.191000e+10
442,Japan,JPN,2019,4.920900e+10
178,Australia,AUS,2019,4.795300e+10
742,Turkiye,TUR,2019,4.141500e+10
760,United Arab Emirates,ARE,2019,3.841330e+10
415,India,IND,2019,3.166100e+10


In [139]:
print(
    "Countries with tourism receipts data:",
    tourism_receipts_2019_df["country_code"].nunique()
)

Countries with tourism receipts data: 24


In [140]:
top_30_country_codes_set = set(top_30_country_codes)

receipts_country_codes_set = set(
    tourism_receipts_2019_df["country_code"]
)

missing_codes = (
    top_30_country_codes_set
    - receipts_country_codes_set
)

missing_codes

{'CAN', 'CHN', 'ESP', 'GBR', 'SGP', 'SWE'}

### Tourism Receipts Assessment

Tourism Receipts was evaluated as a potential KPI to measure the economic value generated by tourism.

However, data availability was limited to 24 of the 30 selected countries. To maintain consistent country coverage across the framework, Tourism Receipts was excluded from the final scoring model.

Tourism Arrivals was retained because it provides complete coverage for all Top 30 countries.

## GDP Growth

### About This Indicator
- **Indicator:** NY.GDP.MKTP.KD.ZG — GDP Growth (Annual %)
- **Why:** Measures how fast a country's economy is growing — countries with stronger economic momentum have more consumers with rising purchasing power, making them more attractive markets for travel investment
- **Years:** 2015–2024 (10 years for trend analysis; 2024 filtered separately for scoring)
- **Scoring note:** HIGHER GDP growth = HIGHER score. Standard direction — no inversion needed.

In [141]:
# Pulling country level data using our identified indicator code.

# Pulling 10 years of data instead of a single year
# This allows us to show trends over time.
# date="2015:2024" — colon means range in the World Bank API

url = f"https://api.worldbank.org/v2/country/{countries_joined}/indicator/NY.GDP.MKTP.KD.ZG"

params = {
    "format":   "json",
    "date":     "2015:2024",
    "per_page": 500
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json()[0])

200
{'page': 1, 'pages': 1, 'per_page': 500, 'total': 300, 'sourceid': '2', 'lastupdated': '2026-04-08'}


In [142]:
print(response.json()[1][0])

{'indicator': {'id': 'NY.GDP.MKTP.KD.ZG', 'value': 'GDP growth (annual %)'}, 'country': {'id': 'AE', 'value': 'United Arab Emirates'}, 'countryiso3code': 'ARE', 'date': '2024', 'value': 3.99180705485398, 'unit': '', 'obs_status': '', 'decimal': 1}


In [143]:
# Converting to a dataframe.
# Year is important because we dont want to have multiple rows per country now

gdp_growth_records = response.json()[1]

# Creating empty list to build the dataframe.
rows = []

for record in gdp_growth_records:
    rows.append({
        "country":    record["country"]["value"],
        "year":       int(record["date"]),
        "gdp_growth": record["value"]
    })

# Full 10 year dataset — can be used for trend visualisations in presentation
gdp_growth_trend_df = pd.DataFrame(rows)

print("Shape:", gdp_growth_trend_df.shape)
print()
gdp_growth_trend_df

Shape: (300, 3)



,country,year,gdp_growth
0,United Arab Emirates,2024,3.991807
1,United Arab Emirates,2023,4.301182
2,United Arab Emirates,2022,7.514484
3,United Arab Emirates,2021,4.552786
4,United Arab Emirates,2020,-8.693426
...,...,...,...
295,United States,2019,2.583825
296,United States,2018,2.966505
297,United States,2017,2.457622
298,United States,2016,1.819451


In [144]:
# Exploring the dataframe

print("Shape of dataset:", gdp_growth_trend_df.shape)
print()
print("Number of rows:",    gdp_growth_trend_df.shape[0])
print("Number of columns:", gdp_growth_trend_df.shape[1])

Shape of dataset: (300, 3)

Number of rows: 300
Number of columns: 3


In [145]:
print("First 10 rows:")
gdp_growth_trend_df.head(10)

First 10 rows:


,country,year,gdp_growth
0,United Arab Emirates,2024,3.991807
1,United Arab Emirates,2023,4.301182
2,United Arab Emirates,2022,7.514484
3,United Arab Emirates,2021,4.552786
4,United Arab Emirates,2020,-8.693426
5,United Arab Emirates,2019,1.271342
6,United Arab Emirates,2018,1.537167
7,United Arab Emirates,2017,-1.061377
8,United Arab Emirates,2016,5.657933
9,United Arab Emirates,2015,7.087673


In [146]:
print("Last 10 rows:")
gdp_growth_trend_df.tail(10)

Last 10 rows:


,country,year,gdp_growth
290,United States,2024,2.793001
291,United States,2023,2.887556
292,United States,2022,2.512375
293,United States,2021,6.055053
294,United States,2020,-2.163029
295,United States,2019,2.583825
296,United States,2018,2.966505
297,United States,2017,2.457622
298,United States,2016,1.819451
299,United States,2015,2.945550


In [147]:
print("Data types:")
print(gdp_growth_trend_df.dtypes)

Data types:
country        object
year            int64
gdp_growth    float64
dtype: object


In [148]:
# Checking for missing values
# We check both the count and the percentage of nulls per column

print("Missing values per column:")
print(gdp_growth_trend_df.isnull().sum())
print()
print("Missing values as % of total rows:")
print((gdp_growth_trend_df.isnull().sum() / len(gdp_growth_trend_df) * 100).round(1))

Missing values per column:
country       0
year          0
gdp_growth    0
dtype: int64

Missing values as % of total rows:
country       0.0
year          0.0
gdp_growth    0.0
dtype: float64


In [149]:
# Checking duplicates

duplicate_count = gdp_growth_trend_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [150]:
# Showing the country list and year list

print("Country list:")
print(sorted(gdp_growth_trend_df["country"].unique().tolist()))
print()
print("Year list:")
print(sorted(gdp_growth_trend_df["year"].unique().tolist()))

Country list:
['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea, Rep.', 'Mexico', 'Netherlands', 'Poland', 'Russian Federation', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'Thailand', 'Turkiye', 'United Arab Emirates', 'United Kingdom', 'United States']

Year list:
[2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]


In [151]:
# Check how many records exist per country
# We expect exactly 10 records per country (one per year)
# If a country has fewer than 10 it means some years are missing

print("Record count per country:")
gdp_growth_trend_df["country"].value_counts().sort_index()

Record count per country:


country
Argentina               10
Australia               10
Austria                 10
Belgium                 10
Brazil                  10
Canada                  10
China                   10
France                  10
Germany                 10
India                   10
Indonesia               10
Ireland                 10
Israel                  10
Italy                   10
Japan                   10
Korea, Rep.             10
Mexico                  10
Netherlands             10
Poland                  10
Russian Federation      10
Saudi Arabia            10
Singapore               10
Spain                   10
Sweden                  10
Switzerland             10
Thailand                10
Turkiye                 10
United Arab Emirates    10
United Kingdom          10
United States           10
Name: count, dtype: int64

In [152]:
# Cleaning the data

# Checking which country&year combinations will be dropped due to missing values
# We document this before removing anything so we can mention it in our assumptions. Ex: Saudi Arabia 2013 — no data available

missing_records = gdp_growth_trend_df[
    gdp_growth_trend_df["gdp_growth"].isnull()
][["country", "year"]]

print("Total records to be dropped:", len(missing_records))
print()
print(missing_records.to_string(index=False))

Total records to be dropped: 0

Empty DataFrame
Columns: [country, year]
Index: []


In [153]:
# Creating a new cleaned DataFrame rather than modifying the original

# Removing rows where gdp_growth is null
gdp_growth_clean_df = gdp_growth_trend_df.dropna(
    subset=["gdp_growth"]
)

# Removing any duplicate rows
gdp_growth_clean_df = gdp_growth_clean_df.drop_duplicates()

# Reseting the index after dropping rows
gdp_growth_clean_df = gdp_growth_clean_df.reset_index(drop=True)

print("Shape before cleaning:", gdp_growth_clean_df.shape)
print("Shape after cleaning:",  gdp_growth_clean_df.shape)

Shape before cleaning: (300, 3)
Shape after cleaning: (300, 3)


In [154]:
# Filtering on 2022 for the scoring model.
# reset_index(drop=True) gives us a clean index starting from 0

gdp_growth_df = gdp_growth_clean_df[
    gdp_growth_clean_df["year"] == 2024
].reset_index(drop=True)

print("Shape of 2024 dataset:", gdp_growth_df.shape)
print()
gdp_growth_df

Shape of 2024 dataset: (30, 3)



,country,year,gdp_growth
0,United Arab Emirates,2024,3.991807
1,Argentina,2024,-1.342931
2,Australia,2024,1.373408
3,Austria,2024,-0.659090
4,Belgium,2024,1.070454
5,Brazil,2024,3.419315
6,Canada,2024,1.554795
7,Switzerland,2024,1.302333
8,China,2024,4.977357
9,Germany,2024,-0.495852


In [155]:
# Final view of 2024 dataset sorted by GDP growth (highest to lowest)
# This gives us a first look at which countries rank highest for economic momentum
# Full ranking will be part of the EDA section

print("Countries ranked by GDP growth (2024):")
gdp_growth_df.sort_values(
    "gdp_growth", ascending=False
).reset_index(drop=True)

Countries ranked by GDP growth (2024):


,country,year,gdp_growth
0,India,2024,6.494766
1,Indonesia,2024,5.030345
2,China,2024,4.977357
3,Singapore,2024,4.388024
4,Russian Federation,2024,4.344375
5,United Arab Emirates,2024,3.991807
6,Spain,2024,3.455254
7,Brazil,2024,3.419315
8,Turkiye,2024,3.327623
9,Poland,2024,3.028416


## Digital Readiness

### About This Indicator
- **Indicator:** IT.NET.USER.ZS — Individuals using the Internet (% of population)
- **Why:** Measures what percentage of a country's population has internet access — the digital addressable audience for a travel platform
- **Years:** 2013–2022 (10 years for trend analysis; 2022 filtered separately for scoring)
- **Scoring note:** Higher internet penetration = higher score (standard direction)

In [156]:
# Pulling country level data using our identified indicator code.

# Pulling 10 years of data instead of a single year
# This allows us to show trends over time.
# date="2013:2022" — colon means range in the World Bank API

url = f"https://api.worldbank.org/v2/country/{countries_joined}/indicator/IT.NET.USER.ZS"

params = {
    "format":   "json",
    "date":     "2013:2022",
    "per_page": 500
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json()[0])

200
{'page': 1, 'pages': 1, 'per_page': 500, 'total': 300, 'sourceid': '2', 'lastupdated': '2026-04-08'}


In [157]:
print(response.json()[1][0])

{'indicator': {'id': 'IT.NET.USER.ZS', 'value': 'Individuals using the Internet (% of population)'}, 'country': {'id': 'AE', 'value': 'United Arab Emirates'}, 'countryiso3code': 'ARE', 'date': '2022', 'value': 100, 'unit': '', 'obs_status': '', 'decimal': 0}


In [158]:
# Converting to a dataframe.
# Year is important because we dont want to have multiple rows per country now

digital_readiness_records = response.json()[1]

# Creating empty list to build the dataframe.
rows = []

for record in digital_readiness_records:
    rows.append({
        "country":            record["country"]["value"],
        "year":               int(record["date"]),
        "internet_users_pct": record["value"]
    })

digital_readiness_trend_df = pd.DataFrame(rows)

print("Shape:", digital_readiness_trend_df.shape)
print()
digital_readiness_trend_df.head(10)

Shape: (300, 3)



,country,year,internet_users_pct
0,United Arab Emirates,2022,100.000000
1,United Arab Emirates,2021,100.000000
2,United Arab Emirates,2020,100.000000
3,United Arab Emirates,2019,99.149998
4,United Arab Emirates,2018,98.450002
5,United Arab Emirates,2017,94.819923
6,United Arab Emirates,2016,90.600007
7,United Arab Emirates,2015,90.500000
8,United Arab Emirates,2014,90.400011
9,United Arab Emirates,2013,88.000000


In [159]:
# Exploring the dataframe

print("Shape of dataset:", digital_readiness_trend_df.shape)
print()
print("Number of rows:",    digital_readiness_trend_df.shape[0])
print("Number of columns:", digital_readiness_trend_df.shape[1])


Shape of dataset: (300, 3)

Number of rows: 300
Number of columns: 3


In [160]:
print("First 10 rows:")
digital_readiness_trend_df.head(10)

First 10 rows:


,country,year,internet_users_pct
0,United Arab Emirates,2022,100.000000
1,United Arab Emirates,2021,100.000000
2,United Arab Emirates,2020,100.000000
3,United Arab Emirates,2019,99.149998
4,United Arab Emirates,2018,98.450002
5,United Arab Emirates,2017,94.819923
6,United Arab Emirates,2016,90.600007
7,United Arab Emirates,2015,90.500000
8,United Arab Emirates,2014,90.400011
9,United Arab Emirates,2013,88.000000


In [161]:
print("Last 10 rows:")
digital_readiness_trend_df.tail(10)

Last 10 rows:


,country,year,internet_users_pct
290,United States,2022,92.728500
291,United States,2021,91.268402
292,United States,2020,90.344704
293,United States,2019,89.430285
294,United States,2018,88.498903
295,United States,2017,87.274889
296,United States,2016,85.544421
297,United States,2015,74.554202
298,United States,2014,73.000000
299,United States,2013,71.400000


In [162]:
print("Data types:")
print(digital_readiness_trend_df.dtypes)

Data types:
country                object
year                    int64
internet_users_pct    float64
dtype: object


In [163]:
# Checking for missing values
# We check both the count and the percentage of nulls per column

print("Missing values per column:")
print(digital_readiness_trend_df.isnull().sum())
print()
print("Missing values as % of total rows:")
print((digital_readiness_trend_df.isnull().sum() / len(digital_readiness_trend_df) * 100).round(1))

Missing values per column:
country               0
year                  0
internet_users_pct    0
dtype: int64

Missing values as % of total rows:
country               0.0
year                  0.0
internet_users_pct    0.0
dtype: float64


In [164]:
# Checking duplicates

duplicate_count = digital_readiness_trend_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [165]:
# Showing the country list and year list

print("Country list:")
print(sorted(digital_readiness_trend_df["country"].unique().tolist()))
print()
print("Year list:")
print(sorted(digital_readiness_trend_df["year"].unique().tolist()))

Country list:
['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea, Rep.', 'Mexico', 'Netherlands', 'Poland', 'Russian Federation', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'Thailand', 'Turkiye', 'United Arab Emirates', 'United Kingdom', 'United States']

Year list:
[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [166]:
# Check how many records exist per country
# We expect exactly 10 records per country (one per year)
# If a country has fewer than 10 it means some years are missing

print("Record count per country:")
digital_readiness_trend_df["country"].value_counts().sort_index()

Record count per country:


country
Argentina               10
Australia               10
Austria                 10
Belgium                 10
Brazil                  10
Canada                  10
China                   10
France                  10
Germany                 10
India                   10
Indonesia               10
Ireland                 10
Israel                  10
Italy                   10
Japan                   10
Korea, Rep.             10
Mexico                  10
Netherlands             10
Poland                  10
Russian Federation      10
Saudi Arabia            10
Singapore               10
Spain                   10
Sweden                  10
Switzerland             10
Thailand                10
Turkiye                 10
United Arab Emirates    10
United Kingdom          10
United States           10
Name: count, dtype: int64

In [167]:
# Cleaning the data

# Checking which country&year combinations will be dropped due to missing values
# We document this before removing anything so we can mention it in our assumptions. Ex: Saudi Arabia 2013 — no data available

missing_records = digital_readiness_trend_df[
    digital_readiness_trend_df["internet_users_pct"].isnull()
][["country", "year"]]

print("Total records to be dropped:", len(missing_records))
print()
print(missing_records.to_string(index=False))

Total records to be dropped: 0

Empty DataFrame
Columns: [country, year]
Index: []


In [168]:
digital_readiness_clean_df = digital_readiness_trend_df.copy()

In [169]:
# Filtering on 2022 for the scoring model.
# reset_index(drop=True) gives us a clean index starting from 0

digital_readiness_df = digital_readiness_clean_df[
    digital_readiness_clean_df["year"] == 2022
].reset_index(drop=True)

print("Shape of 2022 dataset:", digital_readiness_df.shape)
print()
digital_readiness_df

Shape of 2022 dataset: (30, 3)



,country,year,internet_users_pct
0,United Arab Emirates,2022,100.000000
1,Argentina,2022,88.375357
2,Australia,2022,96.119400
3,Austria,2022,93.614091
4,Belgium,2022,94.007831
5,Brazil,2022,80.527751
6,Canada,2022,94.000000
7,Switzerland,2022,96.452797
8,China,2022,75.611316
9,Germany,2022,91.629844


In [170]:
# Final view of 2022 dataset sorted by internet penetration (highest to lowest)
# This gives us a first look at which countries rank highest for digital readiness
# Full ranking will be part of the EDA section

print("Countries ranked by internet penetration (2022):")
digital_readiness_df.sort_values(
    "internet_users_pct", ascending=False
).reset_index(drop=True)

Countries ranked by internet penetration (2022):


,country,year,internet_users_pct
0,United Arab Emirates,2022,100.000000
1,Saudi Arabia,2022,100.000000
2,"Korea, Rep.",2022,97.168553
3,Switzerland,2022,96.452797
4,Australia,2022,96.119400
5,Singapore,2022,95.953865
6,United Kingdom,2022,95.465302
7,Ireland,2022,95.036903
8,Sweden,2022,95.009703
9,Spain,2022,94.485543


## Price Competitiveness

### About This Indicator
- **Indicator:** PA.NUS.PPP — PPP Conversion Factor, GDP (LCU per international $)
- **Why:** Measures how expensive or cheap a country is relative to the US dollar — a lower value means the country is more affordable for international travelers
- **Years:** 2013–2022 (10 years for trend analysis; 2022 filtered separately for scoring)
- **Scoring note:** LOWER PPP = MORE price competitive = HIGHER score. This indicator must be **inverted** during scoring.

In [171]:
# Pulling price competitiveness data from World Bank API
# date="2013:2022": colon means range in the World Bank API
# per_page=500: ensures all records returned in one page

data_url = f"https://api.worldbank.org/v2/country/{countries_joined}/indicator/PA.NUS.PPP"

data_params = {
    "format":   "json",
    "date":     "2013:2022",
    "per_page": 500
}

data_response = requests.get(data_url, params=data_params)

print("Status code:", data_response.status_code)
print()
print("Metadata:")
print(data_response.json()[0])

Status code: 200

Metadata:
{'page': 1, 'pages': 1, 'per_page': 500, 'total': 300, 'sourceid': '2', 'lastupdated': '2026-04-08'}


In [172]:
print("First raw record:")
print(data_response.json()[1][0])

First raw record:
{'indicator': {'id': 'PA.NUS.PPP', 'value': 'PPP conversion factor, GDP (LCU per international $)'}, 'country': {'id': 'AE', 'value': 'United Arab Emirates'}, 'countryiso3code': 'ARE', 'date': '2022', 'value': 2.48316366052119, 'unit': '', 'obs_status': '', 'decimal': 2}


In [173]:
# Converting to a dataframe.
# Year is important because we dont want to have multiple rows per country now

price_records = data_response.json()[1]

rows = []

for record in price_records:
    rows.append({
        "country":    record["country"]["value"],
        "year":       int(record["date"]),
        "ppp_factor": record["value"]
    })

# Converting list of dictionaries to a DataFrame
price_competitiveness_trend_df = pd.DataFrame(rows)

print("Shape:", price_competitiveness_trend_df.shape)
print()
price_competitiveness_trend_df.head(10)

Shape: (300, 3)



,country,year,ppp_factor
0,United Arab Emirates,2022,2.483164
1,United Arab Emirates,2021,2.362567
2,United Arab Emirates,2020,2.088984
3,United Arab Emirates,2019,2.113723
4,United Arab Emirates,2018,2.235159
5,United Arab Emirates,2017,2.285250
6,United Arab Emirates,2016,2.241749
7,United Arab Emirates,2015,2.229247
8,United Arab Emirates,2014,2.213517
9,United Arab Emirates,2013,2.234191


In [174]:
# Exploring the dataframe

print("Shape of dataset:", price_competitiveness_trend_df.shape)
print()
print("Number of rows:",    price_competitiveness_trend_df.shape[0])
print("Number of columns:", price_competitiveness_trend_df.shape[1])


Shape of dataset: (300, 3)

Number of rows: 300
Number of columns: 3


In [175]:
print("First 10 rows:")
price_competitiveness_trend_df.head(10)

First 10 rows:


,country,year,ppp_factor
0,United Arab Emirates,2022,2.483164
1,United Arab Emirates,2021,2.362567
2,United Arab Emirates,2020,2.088984
3,United Arab Emirates,2019,2.113723
4,United Arab Emirates,2018,2.235159
5,United Arab Emirates,2017,2.285250
6,United Arab Emirates,2016,2.241749
7,United Arab Emirates,2015,2.229247
8,United Arab Emirates,2014,2.213517
9,United Arab Emirates,2013,2.234191


In [176]:
print("Last 10 rows:")
price_competitiveness_trend_df.head(10)

Last 10 rows:


,country,year,ppp_factor
0,United Arab Emirates,2022,2.483164
1,United Arab Emirates,2021,2.362567
2,United Arab Emirates,2020,2.088984
3,United Arab Emirates,2019,2.113723
4,United Arab Emirates,2018,2.235159
5,United Arab Emirates,2017,2.285250
6,United Arab Emirates,2016,2.241749
7,United Arab Emirates,2015,2.229247
8,United Arab Emirates,2014,2.213517
9,United Arab Emirates,2013,2.234191


In [177]:
print("Data types:")
print(price_competitiveness_trend_df.dtypes)

Data types:
country        object
year            int64
ppp_factor    float64
dtype: object


In [178]:
# Checking for missing values
# We check both the count and the percentage of nulls per column

print("Missing values per column:")
print(price_competitiveness_trend_df.isnull().sum())
print()
print("Missing values as % of total rows:")
print((price_competitiveness_trend_df.isnull().sum() / len(price_competitiveness_trend_df) * 100).round(1))

Missing values per column:
country       0
year          0
ppp_factor    0
dtype: int64

Missing values as % of total rows:
country       0.0
year          0.0
ppp_factor    0.0
dtype: float64


In [179]:
# Checking duplicates

duplicate_count_ppp = price_competitiveness_trend_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count_ppp)

Number of duplicate rows: 0


In [180]:
# Showing the country list and year list

print("Country list:")
print(sorted(price_competitiveness_trend_df["country"].unique().tolist()))
print()
print("Year list:")
print(sorted(price_competitiveness_trend_df["year"].unique().tolist()))

Country list:
['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea, Rep.', 'Mexico', 'Netherlands', 'Poland', 'Russian Federation', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'Thailand', 'Turkiye', 'United Arab Emirates', 'United Kingdom', 'United States']

Year list:
[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [181]:
# Checking how many records exist per country
# We expect exactly 10 records per country (one per year)
# If a country has fewer than 10 it means some years are missing

print("Record count per country:")
price_competitiveness_trend_df["country"].value_counts().sort_index()

Record count per country:


country
Argentina               10
Australia               10
Austria                 10
Belgium                 10
Brazil                  10
Canada                  10
China                   10
France                  10
Germany                 10
India                   10
Indonesia               10
Ireland                 10
Israel                  10
Italy                   10
Japan                   10
Korea, Rep.             10
Mexico                  10
Netherlands             10
Poland                  10
Russian Federation      10
Saudi Arabia            10
Singapore               10
Spain                   10
Sweden                  10
Switzerland             10
Thailand                10
Turkiye                 10
United Arab Emirates    10
United Kingdom          10
United States           10
Name: count, dtype: int64

In [182]:
# Looking at the range of PPP values to understand what we are working with
# This is specific to PPP — unlike internet penetration (always 0-100%)
# PPP values vary widely across countries and currencies. For example: India might be 22 while Switzerland might be 1
# This wide range is why normalisation is important.

print("PPP factor range by country (2022 preview):")
price_competitiveness_trend_df[
    price_competitiveness_trend_df["year"] == 2022
].sort_values("ppp_factor")[["country", "ppp_factor"]]

PPP factor range by country (2022 preview):


,country,ppp_factor
100,Spain,0.559260
170,Italy,0.591290
120,United Kingdom,0.633175
110,France,0.682729
30,Austria,0.687544
90,Germany,0.688423
40,Belgium,0.698153
210,Netherlands,0.714036
150,Ireland,0.721422
250,Singapore,0.869998


In [183]:
# Cleaning the data

# Checking which country & year combinations will be dropped due to missing values
# We document this before removing anything so we can mention it in our assumptions. Ex: Saudi Arabia 2013 — no data available

missing_records_ppp = price_competitiveness_trend_df[
    price_competitiveness_trend_df["ppp_factor"].isnull()
][["country", "year"]]

print("Total records to be dropped:", len(missing_records_ppp))
print()
print(missing_records_ppp.to_string(index=False))

Total records to be dropped: 0

Empty DataFrame
Columns: [country, year]
Index: []


In [184]:
# Creating a new cleaned DataFrame rather than modifying the original

# Removing rows where ppp_factor is null
price_competitiveness_clean_df = price_competitiveness_trend_df.dropna(
    subset=["ppp_factor"]
)

# Removing any duplicate rows
price_competitiveness_clean_df = price_competitiveness_clean_df.drop_duplicates()

# Reseting the index after dropping rows
price_competitiveness_clean_df = price_competitiveness_clean_df.reset_index(drop=True)

print("Shape before cleaning:", price_competitiveness_trend_df.shape)
print("Shape after cleaning:",  price_competitiveness_clean_df.shape)

Shape before cleaning: (300, 3)
Shape after cleaning: (300, 3)


In [185]:
# Filtering on 2022 for the scoring model.
# reset_index(drop=True) gives us a clean index starting from 0

price_competitiveness_df = price_competitiveness_clean_df[
    price_competitiveness_clean_df["year"] == 2022
].reset_index(drop=True)

print("Shape of 2022 dataset:", price_competitiveness_df.shape)
print()
price_competitiveness_df

Shape of 2022 dataset: (30, 3)



,country,year,ppp_factor
0,United Arab Emirates,2022,2.483164
1,Argentina,2022,61.179647
2,Australia,2022,1.358727
3,Austria,2022,0.687544
4,Belgium,2022,0.698153
5,Brazil,2022,2.411273
6,Canada,2022,1.145083
7,Switzerland,2022,0.986913
8,China,2022,3.794041
9,Germany,2022,0.688423


In [186]:
# Final view of 2022 dataset sorted by PPP factor (lowest to highest)
# Lowest PPP = most affordable = most price competitive for travelers
# This gives us a first look at which countries rank highest for price competitiveness

print("Countries ranked by price competitiveness (2022):")
print("(Lower PPP factor = more affordable = more price competitive)")
print()
price_competitiveness_df.sort_values(
    "ppp_factor", ascending=True
).reset_index(drop=True)

Countries ranked by price competitiveness (2022):
(Lower PPP factor = more affordable = more price competitive)



,country,year,ppp_factor
0,Spain,2022,0.559260
1,Italy,2022,0.591290
2,United Kingdom,2022,0.633175
3,France,2022,0.682729
4,Austria,2022,0.687544
5,Germany,2022,0.688423
6,Belgium,2022,0.698153
7,Netherlands,2022,0.714036
8,Ireland,2022,0.721422
9,Singapore,2022,0.869998


## Urban Population %

### About This Indicator
- **Indicator:** SP.URB.TOTL.IN.ZS — Urban Population (% of Total Population)
- **Why:** Measures what share of a country's population lives in cities — urban populations are more likely to book travel online and have higher travel spending, making this a useful supporting indicator for a digital travel platform
- **Years:** 2013–2022 (10 years for trend analysis; 2022 filtered separately for scoring)
- **Scoring note:** This is an optional supporting indicator — it is not included in the core Market Attractiveness Score but supports EDA analysis and presentation context.

In [187]:
# Pulling country level data using our identified indicator code.

# Pulling 10 years of data instead of a single year
# This allows us to show trends over time.
# date="2013:2022" — colon means range in the World Bank API

url = f"https://api.worldbank.org/v2/country/{countries_joined}/indicator/SP.URB.TOTL.IN.ZS"

params = {
    "format":   "json",
    "date":     "2013:2022",
    "per_page": 500
}

response = requests.get(url, params=params)

print(response.status_code)
print(response.json()[0])

200
{'page': 1, 'pages': 1, 'per_page': 500, 'total': 300, 'sourceid': '2', 'lastupdated': '2026-04-08'}


In [188]:
print(response.json()[1][0])

{'indicator': {'id': 'SP.URB.TOTL.IN.ZS', 'value': 'Urban population (% of total population)'}, 'country': {'id': 'AE', 'value': 'United Arab Emirates'}, 'countryiso3code': 'ARE', 'date': '2022', 'value': 85.5147347682487, 'unit': '', 'obs_status': '', 'decimal': 0}


In [189]:
# Converting to a dataframe.
# Year is important because we dont want to have multiple rows per country now

urban_population_records = response.json()[1]

# Creating empty list to build the dataframe.
rows = []

for record in urban_population_records:
    rows.append({
        "country":          record["country"]["value"],
        "year":             int(record["date"]),
        "urban_pop_pct":    record["value"]
    })

# Full 10 year dataset — can be used for trend visualisations in presentation
urban_population_trend_df = pd.DataFrame(rows)

print("Shape:", urban_population_trend_df.shape)
print()
urban_population_trend_df

Shape: (300, 3)



,country,year,urban_pop_pct
0,United Arab Emirates,2022,85.514735
1,United Arab Emirates,2021,85.357191
2,United Arab Emirates,2020,85.195526
3,United Arab Emirates,2019,85.029826
4,United Arab Emirates,2018,84.860181
...,...,...,...
295,United States,2017,80.401868
296,United States,2016,80.497665
297,United States,2015,80.570799
298,United States,2014,80.624223


In [190]:
# Exploring the dataframe

print("Shape of dataset:", urban_population_trend_df.shape)
print()
print("Number of rows:",    urban_population_trend_df.shape[0])
print("Number of columns:", urban_population_trend_df.shape[1])

Shape of dataset: (300, 3)

Number of rows: 300
Number of columns: 3


In [191]:
print("First 10 rows:")
urban_population_trend_df.head(10)

First 10 rows:


,country,year,urban_pop_pct
0,United Arab Emirates,2022,85.514735
1,United Arab Emirates,2021,85.357191
2,United Arab Emirates,2020,85.195526
3,United Arab Emirates,2019,85.029826
4,United Arab Emirates,2018,84.860181
5,United Arab Emirates,2017,84.686676
6,United Arab Emirates,2016,84.509399
7,United Arab Emirates,2015,84.328438
8,United Arab Emirates,2014,84.143881
9,United Arab Emirates,2013,83.955814


In [192]:
print("Last 10 rows:")
urban_population_trend_df.tail(10)

Last 10 rows:


,country,year,urban_pop_pct
290,United States,2022,80.032498
291,United States,2021,80.007477
292,United States,2020,79.997452
293,United States,2019,80.130479
294,United States,2018,80.280457
295,United States,2017,80.401868
296,United States,2016,80.497665
297,United States,2015,80.570799
298,United States,2014,80.624223
299,United States,2013,80.660889


In [193]:
print("Data types:")
print(urban_population_trend_df.dtypes)

Data types:
country           object
year               int64
urban_pop_pct    float64
dtype: object


In [194]:
# Checking for missing values
# We check both the count and the percentage of nulls per column

print("Missing values per column:")
print(urban_population_trend_df.isnull().sum())
print()
print("Missing values as % of total rows:")
print((urban_population_trend_df.isnull().sum() / len(urban_population_trend_df) * 100).round(1))

Missing values per column:
country          0
year             0
urban_pop_pct    0
dtype: int64

Missing values as % of total rows:
country          0.0
year             0.0
urban_pop_pct    0.0
dtype: float64


In [195]:
# Checking duplicates

duplicate_count = urban_population_trend_df.duplicated().sum()
print("Number of duplicate rows:", duplicate_count)

Number of duplicate rows: 0


In [196]:
# Showing the country list and year list

print("Country list:")
print(sorted(urban_population_trend_df["country"].unique().tolist()))
print()
print("Year list:")
print(sorted(urban_population_trend_df["year"].unique().tolist()))

Country list:
['Argentina', 'Australia', 'Austria', 'Belgium', 'Brazil', 'Canada', 'China', 'France', 'Germany', 'India', 'Indonesia', 'Ireland', 'Israel', 'Italy', 'Japan', 'Korea, Rep.', 'Mexico', 'Netherlands', 'Poland', 'Russian Federation', 'Saudi Arabia', 'Singapore', 'Spain', 'Sweden', 'Switzerland', 'Thailand', 'Turkiye', 'United Arab Emirates', 'United Kingdom', 'United States']

Year list:
[2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


In [197]:
# Check how many records exist per country
# We expect exactly 10 records per country (one per year)
# If a country has fewer than 10 it means some years are missing

print("Record count per country:")
urban_population_trend_df["country"].value_counts().sort_index()

Record count per country:


country
Argentina               10
Australia               10
Austria                 10
Belgium                 10
Brazil                  10
Canada                  10
China                   10
France                  10
Germany                 10
India                   10
Indonesia               10
Ireland                 10
Israel                  10
Italy                   10
Japan                   10
Korea, Rep.             10
Mexico                  10
Netherlands             10
Poland                  10
Russian Federation      10
Saudi Arabia            10
Singapore               10
Spain                   10
Sweden                  10
Switzerland             10
Thailand                10
Turkiye                 10
United Arab Emirates    10
United Kingdom          10
United States           10
Name: count, dtype: int64

In [198]:
# Cleaning the data

# Checking which country&year combinations will be dropped due to missing values
# We document this before removing anything so we can mention it in our assumptions. Ex: Saudi Arabia 2013 — no data available

missing_records = urban_population_trend_df[
    urban_population_trend_df["urban_pop_pct"].isnull()
][["country", "year"]]

print("Total records to be dropped:", len(missing_records))
print()
print(missing_records.to_string(index=False))

Total records to be dropped: 0

Empty DataFrame
Columns: [country, year]
Index: []


In [199]:
urban_population_clean_df = urban_population_trend_df.copy()

In [200]:
# Filtering on 2022 for the scoring model.
# reset_index(drop=True) gives us a clean index starting from 0

urban_population_df = urban_population_clean_df[
    urban_population_clean_df["year"] == 2022
].reset_index(drop=True)

print("Shape of 2022 dataset:", urban_population_df.shape)
print()
urban_population_df

Shape of 2022 dataset: (30, 3)



,country,year,urban_pop_pct
0,United Arab Emirates,2022,85.514735
1,Argentina,2022,92.112838
2,Australia,2022,87.334327
3,Austria,2022,68.998085
4,Belgium,2022,87.489200
5,Brazil,2022,87.349556
6,Canada,2022,82.350090
7,Switzerland,2022,84.920627
8,China,2022,65.217636
9,Germany,2022,81.786785


## Optional World Bank KPI Data Collection

Next, we collect additional World Bank KPIs for the Top 30 GDP countries.

These indicators add more context to the market attractiveness framework:

1. GDP per capita: economic strength per person
2. Population: market size
3. Urban Population %: urbanization and accessibility
4. Mobile Subscriptions: digital connectivity

All indicators will be filtered to the Top 30 GDP countries.

In [201]:
additional_kpis = {
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",
    "SP.POP.TOTL": "population",
    "SP.URB.TOTL.IN.ZS": "urban_population_pct",
    "IT.CEL.SETS.P2": "mobile_subscriptions_per_100"
}

In [202]:
def pull_world_bank_kpi(indicator_code, value_column_name, start_year=2019, end_year=2024):
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator_code}"
    
    params = {
        "format": "json",
        "date": f"{start_year}:{end_year}",
        "per_page": 20000
    }
    
    response = requests.get(url, params=params)
    
    print(indicator_code, "status code:", response.status_code)
    
    raw_data = response.json()
    raw_df = pd.DataFrame(raw_data[1])
    
    df = raw_df[["country", "countryiso3code", "date", "value"]].copy()
    
    df["country"] = df["country"].apply(lambda x: x["value"])
    
    df = df.rename(columns={
        "countryiso3code": "country_code",
        "date": "year",
        "value": value_column_name
    })
    
    df["year"] = df["year"].astype(int)
    
    df = df[df["country_code"].apply(is_valid_country)].copy()
    
    df = df[df["country_code"].isin(top_30_country_codes)].copy()
    
    return df

In [203]:
gdp_per_capita_df = pull_world_bank_kpi(
    "NY.GDP.PCAP.CD",
    "gdp_per_capita_usd"
)

gdp_per_capita_df.head()

NY.GDP.PCAP.CD status code: 200


,country,country_code,year,gdp_per_capita_usd
336,Argentina,ARG,2024,13969.783660
337,Argentina,ARG,2023,14261.846567
338,Argentina,ARG,2022,13962.189409
339,Argentina,ARG,2021,10738.017922
340,Argentina,ARG,2020,8535.599380


In [204]:
population_df = pull_world_bank_kpi(
    "SP.POP.TOTL",
    "population"
)

population_df.head()

SP.POP.TOTL status code: 200


,country,country_code,year,population
336,Argentina,ARG,2024,45696159.0
337,Argentina,ARG,2023,45538401.0
338,Argentina,ARG,2022,45407904.0
339,Argentina,ARG,2021,45312281.0
340,Argentina,ARG,2020,45191965.0


In [205]:
urban_population_df = pull_world_bank_kpi(
    "SP.URB.TOTL.IN.ZS",
    "urban_population_pct"
)

urban_population_df.head()


SP.URB.TOTL.IN.ZS status code: 200


,country,country_code,year,urban_population_pct
336,Argentina,ARG,2024,92.274232
337,Argentina,ARG,2023,92.194513
338,Argentina,ARG,2022,92.112838
339,Argentina,ARG,2021,92.029278
340,Argentina,ARG,2020,91.943903


In [206]:
mobile_subscriptions_df = pull_world_bank_kpi(
    "IT.CEL.SETS.P2",
    "mobile_subscriptions_per_100"
)

mobile_subscriptions_df.head()

IT.CEL.SETS.P2 status code: 200


,country,country_code,year,mobile_subscriptions_per_100
336,Argentina,ARG,2024,140.234979
337,Argentina,ARG,2023,137.706087
338,Argentina,ARG,2022,131.483556
339,Argentina,ARG,2021,130.352800
340,Argentina,ARG,2020,121.180613


In [207]:
# After pulling the additional KPIs, we check the country coverage for each indicator.

additional_kpi_dfs = {
    "GDP per capita": gdp_per_capita_df,
    "Population": population_df,
    "Urban Population %": urban_population_df,
    "Mobile Subscriptions": mobile_subscriptions_df
}

for name, df in additional_kpi_dfs.items():
    print(name)
    print("Rows:", len(df))
    print("Unique countries:", df["country_code"].nunique())
    print("Missing values:", df.isna().sum().sum())
    print()

GDP per capita
Rows: 180
Unique countries: 30
Missing values: 0

Population
Rows: 180
Unique countries: 30
Missing values: 0

Urban Population %
Rows: 180
Unique countries: 30
Missing values: 0

Mobile Subscriptions
Rows: 180
Unique countries: 30
Missing values: 2



In [208]:
# Next, we keep the latest available year for each KPI.
# The target year is 2024. If data is not available for 2024, we will review coverage before deciding whether to use the latest available year or exclude the KPI.

for name, df in additional_kpi_dfs.items():
    print(name)
    print(df.groupby("year")["country_code"].nunique())
    print()

GDP per capita
year
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
Name: country_code, dtype: int64

Population
year
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
Name: country_code, dtype: int64

Urban Population %
year
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
Name: country_code, dtype: int64

Mobile Subscriptions
year
2019    30
2020    30
2021    30
2022    30
2023    30
2024    30
Name: country_code, dtype: int64



## KPI Coverage Validation

Before proceeding to exploratory analysis, we validate that all selected KPIs provide sufficient coverage for the Top 30 GDP countries.

Complete country coverage is important because missing values can introduce bias into the final market attractiveness ranking.

The selected indicators demonstrate strong data availability and will be used in the subsequent analysis.

In [209]:
kpi_coverage_summary = pd.DataFrame({
    "KPI": [
        "GDP",
        "GDP Growth",
        "Tourism Arrivals",
        "GDP per Capita",
        "Population",
        "Urban Population %",
        "Mobile Subscriptions"
    ],
    "Countries Covered": [
        top_30_gdp_2024_df["country_code"].nunique(),
        gdp_growth_df["country"].nunique(),
        tourism_2019_df["country_code"].nunique(),
        gdp_per_capita_df[gdp_per_capita_df["year"] == 2024]["country_code"].nunique(),
        population_df[population_df["year"] == 2024]["country_code"].nunique(),
        urban_population_df[urban_population_df["year"] == 2024]["country_code"].nunique(),
        mobile_subscriptions_df[mobile_subscriptions_df["year"] == 2024]["country_code"].nunique()
    ]
})

kpi_coverage_summary

,KPI,Countries Covered
0,GDP,30
1,GDP Growth,30
2,Tourism Arrivals,30
3,GDP per Capita,30
4,Population,30
5,Urban Population %,30
6,Mobile Subscriptions,30


## Create the Final Market KPI Dataset

After cleaning each World Bank indicator individually, the selected KPIs are merged into a single dataset.

This consolidated dataset contains one row per country and serves as the primary input for the exploratory data analysis and market attractiveness framework developed in the Notebook 04.

The merge is performed using the country name and, where available, the country code to ensure that each indicator is correctly aligned across all countries.

Before exporting the final dataset, we validate the merge by checking the number of countries and identifying any missing values to confirm that the dataset is ready for analysis.

In [210]:
# 1. Keep one row per country for each KPI

gdp_2024_final_df = top_30_gdp_2024_df[
    ["country", "country_code", "gdp_usd", "year"]
].copy().rename(columns={"year": "gdp_year"})

tourism_arrivals_2019_final_df = tourism_2019_df[
    ["country", "country_code", "tourism_arrivals", "year"]
].copy().rename(columns={"year": "tourism_arrivals_year"})

gdp_growth_2024_final_df = gdp_growth_df[
    ["country", "gdp_growth", "year"]
].copy().rename(columns={"year": "gdp_growth_year"})

internet_2022_final_df = digital_readiness_df[
    ["country", "internet_users_pct", "year"]
].copy().rename(columns={"year": "internet_users_year"})

ppp_2022_final_df = price_competitiveness_df[
    ["country", "ppp_factor", "year"]
].copy().rename(columns={"year": "ppp_factor_year"})

gdp_per_capita_2024_final_df = gdp_per_capita_df[
    gdp_per_capita_df["year"] == 2024
][["country", "country_code", "gdp_per_capita_usd", "year"]].copy().rename(columns={"year": "gdp_per_capita_year"})

population_2024_final_df = population_df[
    population_df["year"] == 2024
][["country", "country_code", "population", "year"]].copy().rename(columns={"year": "population_year"})

urban_population_2024_final_df = urban_population_df[
    urban_population_df["year"] == 2024
][["country", "country_code", "urban_population_pct", "year"]].copy().rename(columns={"year": "urban_population_year"})

mobile_subscriptions_2024_final_df = mobile_subscriptions_df[
    mobile_subscriptions_df["year"] == 2024
][["country", "country_code", "mobile_subscriptions_per_100", "year"]].copy().rename(columns={"year": "mobile_subscriptions_year"})

In [211]:
# 2. Merge everything into one final World Bank KPI dataset

market_kpis_df = (
    gdp_2024_final_df
    .merge(tourism_arrivals_2019_final_df, on=["country", "country_code"], how="left")
    .merge(gdp_growth_2024_final_df, on="country", how="left")
    .merge(internet_2022_final_df, on="country", how="left")
    .merge(ppp_2022_final_df, on="country", how="left")
    .merge(gdp_per_capita_2024_final_df, on=["country", "country_code"], how="left")
    .merge(population_2024_final_df, on=["country", "country_code"], how="left")
    .merge(urban_population_2024_final_df, on=["country", "country_code"], how="left")
    .merge(mobile_subscriptions_2024_final_df, on=["country", "country_code"], how="left")
)

market_kpis_df.head()

,country,country_code,gdp_usd,gdp_year,tourism_arrivals,tourism_arrivals_year,gdp_growth,gdp_growth_year,internet_users_pct,internet_users_year,ppp_factor,ppp_factor_year,gdp_per_capita_usd,gdp_per_capita_year,population,population_year,urban_population_pct,urban_population_year,mobile_subscriptions_per_100,mobile_subscriptions_year
0,United States,USA,2.875096e+13,2024,165478000.0,2019,2.793001,2024,92.728500,2022,1.000000,2022,84534.040784,2024,3.401110e+08,2024,80.124780,2024,113.190482,2024
1,China,CHN,1.874380e+13,2024,162538000.0,2019,4.977357,2024,75.611316,2022,3.794041,2022,13303.148154,2024,1.408975e+09,2024,65.894698,2024,131.753932,2024
2,Germany,DEU,4.685593e+12,2024,39563000.0,2019,-0.495852,2024,91.629844,2022,0.688423,2022,56103.732318,2024,8.351659e+07,2024,82.021143,2024,129.150922,2024
3,Japan,JPN,4.027598e+12,2024,31881000.0,2019,0.104309,2024,84.923431,2022,94.922107,2022,32487.077805,2024,1.239754e+08,2024,92.190172,2024,NaN,2024
4,India,IND,3.909892e+12,2024,17914000.0,2019,6.494766,2024,55.900000,2022,20.489457,2022,2694.737809,2024,1.450936e+09,2024,35.378897,2024,79.352692,2024


In [212]:
# 3. Check the final dataset

print("Final market KPI dataset shape:", market_kpis_df.shape)
print()
print("Number of countries:", market_kpis_df["country_code"].nunique())
print()
print("Missing values by column:")
print(market_kpis_df.isna().sum())

Final market KPI dataset shape: (30, 20)

Number of countries: 30

Missing values by column:
country                         0
country_code                    0
gdp_usd                         0
gdp_year                        0
tourism_arrivals                0
tourism_arrivals_year           0
gdp_growth                      0
gdp_growth_year                 0
internet_users_pct              0
internet_users_year             0
ppp_factor                      0
ppp_factor_year                 0
gdp_per_capita_usd              0
gdp_per_capita_year             0
population                      0
population_year                 0
urban_population_pct            0
urban_population_year           0
mobile_subscriptions_per_100    2
mobile_subscriptions_year       0
dtype: int64


In [213]:
market_kpis_df.to_csv("market_kpis.csv", index=False)

print("market_kpis.csv exported successfully.")

market_kpis.csv exported successfully.


## Data Collection Summary

- A total of seven KPIs were collected from the World Bank API.
- The indicators cover economic strength, economic growth, tourism activity, market size, urbanization, and digital connectivity.
- The dataset is now ready for exploratory data analysis (EDA), where we will examine the distribution of each KPI and identify patterns across countries.

---

### KPIs Collected

**Core Scoring Indicators** — used directly in the Market Attractiveness Score:
- **Tourism Arrivals** (`ST.INT.ARVL`) — 2019 reference year (last full pre-pandemic year)
- **GDP Growth** (`NY.GDP.MKTP.KD.ZG`) — 2015–2024 trend, 2024 for scoring
- **Internet Users / Digital Readiness** (`IT.NET.USER.ZS`) — 2013–2022 trend, 2022 for scoring
- **PPP Conversion Factor / Price Competitiveness** (`PA.NUS.PPP`) — 2013–2022 trend, 2022 for scoring

**Supporting Indicators** — used for EDA context and presentation:
- **GDP per Capita** (`NY.GDP.PCAP.CD`) — 2019–2024
- **Population** (`SP.POP.TOTL`) — 2019–2024
- **Urban Population %** (`SP.URB.TOTL.IN.ZS`) — 2019–2024
- **Mobile Subscriptions** (`IT.CEL.SETS.P2`) — 2019–2024

---

### Scope
- **Countries:** Top 30 by GDP (2024), derived programmatically from World Bank data